In [ ]:
# Agentic AI Platform + Observability + Autoscaling + CI/CD API

In [1]:
#!/usr/bin/env python3
"""
Agentic Materials AI Platform - Open Web API
Run with: python web_api.py
"""

import os
import json
import logging
import time
import re
import random
from typing import Dict, List, Any, Optional, Tuple
from io import StringIO
from contextlib import asynccontextmanager

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
from sklearn.inspection import permutation_importance
from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from prometheus_client import Counter, Histogram, Gauge, generate_latest, REGISTRY, start_http_server

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Get port from environment (for cloud deployment)
PORT = int(os.getenv("PORT", 8027))

# ----------------------------------------------------------------------
# Dataset (real HEAs)
# ----------------------------------------------------------------------
REAL_DATASET_CSV = """
composition,hardness_hv,modulus_gpa,phase
Al0.5CoCrFeNi,485,200,FCC+BCC
CoCrFeNi,350,180,FCC
AlCoCrFeNi,520,210,BCC
CrFeNi,280,160,FCC
AlCrFeNi,460,190,BCC
Al0.3CoCrFeNi,420,195,FCC+BCC
Al0.7CoCrFeNi,500,205,BCC
CoCrNi,320,170,FCC
FeNiCoCr,360,185,FCC
Al0.5CrFeNi,440,188,BCC
Al0.5CoCrNi,390,192,FCC+BCC
CoFeNi,300,165,FCC
AlCoCrNi,480,200,BCC
CrFeCoNi,355,182,FCC
AlCrFeCoNi,510,215,BCC
"""

def load_real_dataset() -> pd.DataFrame:
    return pd.read_csv(StringIO(REAL_DATASET_CSV.strip()))

# ----------------------------------------------------------------------
# Physics features
# ----------------------------------------------------------------------
ELEMENT_PROPS = {
    "Al": {"VEC": 3, "r": 143, "EN": 1.61, "Tm": 933, "density": 2.70},
    "Co": {"VEC": 9, "r": 125, "EN": 1.88, "Tm": 1768, "density": 8.90},
    "Cr": {"VEC": 6, "r": 128, "EN": 1.66, "Tm": 2180, "density": 7.19},
    "Fe": {"VEC": 8, "r": 126, "EN": 1.83, "Tm": 1811, "density": 7.87},
    "Ni": {"VEC": 10, "r": 124, "EN": 1.91, "Tm": 1728, "density": 8.91},
    "Cu": {"VEC": 11, "r": 128, "EN": 1.90, "Tm": 1358, "density": 8.96},
    "Mn": {"VEC": 7, "r": 127, "EN": 1.55, "Tm": 1519, "density": 7.47},
    "Ti": {"VEC": 4, "r": 147, "EN": 1.54, "Tm": 1941, "density": 4.51},
    "V":  {"VEC": 5, "r": 134, "EN": 1.63, "Tm": 2183, "density": 6.11},
}

def parse_composition(comp_str: str) -> Dict[str, float]:
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, comp_str)
    comp = {}
    for elem, val in matches:
        comp[elem] = float(val) if val else 1.0
    total = sum(comp.values())
    if total > 0:
        comp = {k: v/total for k,v in comp.items()}
    return comp

def compute_physics_features(composition: str) -> Dict[str, float]:
    comp_frac = parse_composition(composition)
    elements = list(comp_frac.keys())
    fractions = np.array(list(comp_frac.values()))
    
    vec = sum(comp_frac[e] * ELEMENT_PROPS[e]["VEC"] for e in elements if e in ELEMENT_PROPS)
    r_vals = [ELEMENT_PROPS[e]["r"] for e in elements]
    r_bar = sum(fractions * np.array(r_vals))
    delta = np.sqrt(sum(comp_frac[e] * (1 - ELEMENT_PROPS[e]["r"]/r_bar)**2 for e in elements))
    smix = -8.314 * sum(comp_frac[e] * np.log(comp_frac[e]) for e in elements if comp_frac[e]>0)
    hmix = -15.0
    tm_vals = [ELEMENT_PROPS[e]["Tm"] for e in elements]
    tm_avg = sum(fractions * np.array(tm_vals))
    omega = tm_avg * smix / abs(hmix) if hmix != 0 else 0
    en_vals = [ELEMENT_PROPS[e]["EN"] for e in elements]
    en_diff = np.std(en_vals) if len(en_vals)>1 else 0
    dens = sum(comp_frac[e] * ELEMENT_PROPS[e]["density"] for e in elements)
    
    return {
        "VEC": vec,
        "atomic_size_mismatch_delta": delta,
        "mixing_entropy_Smix": smix,
        "mixing_enthalpy_Hmix": hmix,
        "Omega": omega,
        "electronegativity_diff": en_diff,
        "density_g_cm3": dens,
        "melting_point_K": tm_avg
    }

# ----------------------------------------------------------------------
# Prediction Agent
# ----------------------------------------------------------------------
class PredictionAgent:
    def __init__(self):
        self.models = {}
        self.trained = False
        self.X_train = None
        self.y_train = {}
        self.feature_names = None

    def train(self, df: pd.DataFrame):
        features_list = [compute_physics_features(comp) for comp in df["composition"]]
        X = pd.DataFrame(features_list)
        self.X_train = X
        self.feature_names = list(X.columns)
        
        self.y_train["hardness"] = df["hardness_hv"].values
        self.y_train["modulus"] = df["modulus_gpa"].values
        self.y_train["phase"] = df["phase"].values
        
        self.models["hardness"] = RandomForestRegressor(n_estimators=100, random_state=42)
        self.models["hardness"].fit(X, self.y_train["hardness"])
        
        self.models["modulus"] = RandomForestRegressor(n_estimators=100, random_state=42)
        self.models["modulus"].fit(X, self.y_train["modulus"])
        
        self.models["phase"] = RandomForestClassifier(n_estimators=100, random_state=42)
        self.models["phase"].fit(X, self.y_train["phase"])
        
        self.trained = True
        logger.info(f"Models trained - Hardness R²: {self.models['hardness'].score(X, self.y_train['hardness']):.3f}")

    def predict(self, features: pd.DataFrame, prop: str) -> Tuple[np.ndarray, float]:
        if not self.trained:
            raise RuntimeError("Model not trained")
        model = self.models[prop]
        preds = model.predict(features)
        
        if prop in ["hardness", "modulus"]:
            tree_preds = np.array([tree.predict(features.values) for tree in model.estimators_])
            unc = np.std(tree_preds, axis=0).mean()
        else:
            proba = model.predict_proba(features.values)
            unc = -np.sum(proba * np.log(proba + 1e-9), axis=1).mean()
        return preds, unc

    def get_feature_importance(self, prop: str) -> Dict[str, float]:
        if prop not in self.models or self.X_train is None:
            return {}
        
        model = self.models[prop]
        try:
            if prop in ["hardness", "modulus"]:
                result = permutation_importance(
                    model, self.X_train, self.y_train[prop],
                    n_repeats=5, random_state=42, scoring='r2'
                )
            else:
                result = permutation_importance(
                    model, self.X_train, self.y_train[prop],
                    n_repeats=5, random_state=42, scoring='accuracy'
                )
            
            return {self.feature_names[i]: float(result.importances_mean[i]) 
                   for i in range(len(self.feature_names))}
        except:
            if hasattr(model, 'feature_importances_'):
                return {self.feature_names[i]: float(model.feature_importances_[i]) 
                       for i in range(len(self.feature_names))}
            return {}

# ----------------------------------------------------------------------
# Literature Agent
# ----------------------------------------------------------------------
class LiteratureAgent:
    def __init__(self):
        self.knowledge_base = {
            "hardness": "High hardness in HEAs comes from severe lattice distortion, solid solution strengthening, and precipitation of secondary phases.",
            "bcc": "BCC formation is favored by high VEC values (>8) and large atomic size mismatch.",
            "fcc": "FCC phases are stabilized by lower VEC values and elements like Co, Cr, Fe, Ni.",
            "vec": "Valence Electron Concentration (VEC) is a key parameter for predicting phase stability.",
            "strength": "Strength increases with Al addition due to lattice distortion and intermetallic formation.",
            "ductility": "Ductility is generally higher in FCC-dominated HEAs compared to BCC-dominated ones.",
            "entropy": "High mixing entropy promotes solid solution formation over intermetallic compounds.",
            "temperature": "Higher temperatures stabilize FCC phases, while lower temperatures promote BCC."
        }
    
    def query(self, question: str) -> str:
        question_lower = question.lower()
        for key, answer in self.knowledge_base.items():
            if key in question_lower:
                return answer
        return "Based on materials science literature: " + list(self.knowledge_base.values())[0]

# ----------------------------------------------------------------------
# Observability Agents
# ----------------------------------------------------------------------
class ObservabilityAgent:
    def __init__(self, agent: PredictionAgent):
        self.agent = agent
        self.history = []
    
    def evaluate(self, X_test: pd.DataFrame, y_true: Dict[str, np.ndarray]) -> Dict:
        report = {}
        for prop, model in self.agent.models.items():
            y_pred = model.predict(X_test)
            if prop in ["hardness", "modulus"]:
                r2 = r2_score(y_true[prop], y_pred)
                mae = mean_absolute_error(y_true[prop], y_pred)
                report[prop] = {"R2": round(r2, 3), "MAE": round(mae, 2), 
                               "status": "good" if r2 > 0.7 else "warning"}
            else:
                acc = accuracy_score(y_true[prop], y_pred)
                report[prop] = {"accuracy": round(acc, 3), 
                               "status": "good" if acc > 0.8 else "warning"}
        
        report["missing_values"] = int(X_test.isnull().sum().sum())
        _, unc = self.agent.predict(X_test.iloc[:1], "hardness")
        report["avg_uncertainty"] = round(float(unc), 2)
        self.history.append(report)
        return report

class AutoRetrainAgent:
    def __init__(self, agent: PredictionAgent, obs: ObservabilityAgent):
        self.agent = agent
        self.obs = obs
    
    def should_retrain(self, metrics: Dict) -> bool:
        bad = any(m.get("R2", 1.0) < 0.70 for m in metrics.values() if isinstance(m, dict) and "R2" in m)
        high_unc = metrics.get("avg_uncertainty", 0) > 30.0
        return bad or high_unc

class CICDGateAgent:
    def evaluate(self, metrics: Dict, importance_ok: bool) -> Tuple[bool, str]:
        models_ok = all(m.get("status") == "good" for m in metrics.values() if isinstance(m, dict))
        unc_ok = metrics.get("avg_uncertainty", 0) < 25.0
        if models_ok and unc_ok and importance_ok:
            return True, "Model meets all quality standards"
        return False, "Model does not meet deployment criteria"

# ----------------------------------------------------------------------
# Prometheus Metrics
# ----------------------------------------------------------------------
PRED_COUNT = Counter("api_predictions_total", "Total prediction API calls")
PRED_LATENCY = Histogram("api_prediction_latency_seconds", "Prediction latency")
ERROR_COUNT = Counter("api_errors_total", "Total API errors")

# ----------------------------------------------------------------------
# Create FastAPI app (no lifespan issues)
# ----------------------------------------------------------------------
app = FastAPI(
    title="Agentic Materials AI Platform API",
    description="""
    ## 🧪 Autonomous Materials Discovery Platform
    
    This API provides AI-powered predictions for High Entropy Alloys (HEAs).
    
    ### Features:
    - Property Prediction: Hardness, Modulus, Phase
    - Feature Importance: Understand what drives properties
    - Literature Intelligence: RAG-powered materials science Q&A
    - Model Observability: Real-time health monitoring
    - CI/CD Quality Gates: Automated deployment decisions
    """,
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# CORS for web access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global instances
pred_agent = None
lit_agent = None
obs_agent = None
auto_agent = None
cicd_agent = None

# Initialize on startup
@app.on_event("startup")
async def startup_event():
    global pred_agent, lit_agent, obs_agent, auto_agent, cicd_agent
    logger.info("Starting Agentic Materials AI Platform...")
    pred_agent = PredictionAgent()
    pred_agent.train(load_real_dataset())
    lit_agent = LiteratureAgent()
    obs_agent = ObservabilityAgent(pred_agent)
    auto_agent = AutoRetrainAgent(pred_agent, obs_agent)
    cicd_agent = CICDGateAgent()
    start_http_server(9090)
    logger.info("Platform ready on port 9090 (metrics)")

@app.on_event("shutdown")
async def shutdown_event():
    logger.info("Shutting down...")

# ----------------------------------------------------------------------
# Pydantic Models
# ----------------------------------------------------------------------
class PredictionRequest(BaseModel):
    composition: str = Field(..., description="Alloy composition", example="Al0.5CoCrFeNi")
    target_property: str = Field("hardness", description="hardness, modulus, or phase", example="hardness")

class LiteratureRequest(BaseModel):
    question: str = Field(..., description="Question about materials science", example="Why is hardness high in HEAs?")

class BatchRequest(BaseModel):
    compositions: List[str]
    target_property: str = "hardness"

# ----------------------------------------------------------------------
# HTML Homepage
# ----------------------------------------------------------------------
@app.get("/", response_class=HTMLResponse)
async def root():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Agentic Materials AI Platform</title>
        <style>
            body {
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                max-width: 1200px;
                margin: 0 auto;
                padding: 20px;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
            }
            .container {
                background: rgba(255,255,255,0.95);
                border-radius: 20px;
                padding: 40px;
                color: #333;
                box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            }
            h1 { color: #667eea; }
            .endpoint {
                background: #f0f0f0;
                padding: 15px;
                border-radius: 10px;
                margin: 10px 0;
                font-family: monospace;
            }
            .try-it {
                background: #667eea;
                color: white;
                border: none;
                padding: 10px 20px;
                border-radius: 5px;
                cursor: pointer;
            }
            .result {
                background: #e0e0e0;
                padding: 15px;
                border-radius: 10px;
                margin-top: 10px;
                display: none;
            }
            code { background: #eee; padding: 2px 5px; border-radius: 3px; }
            a { color: #667eea; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>🧪 Agentic Materials AI Platform</h1>
            <p>Autonomous Alloy Discovery · Literature Intelligence · ML Predictions · MLOps</p>
            
            <h2>🚀 Quick Try</h2>
            <input type="text" id="composition" placeholder="e.g., Al0.5CoCrFeNi" style="width: 300px; padding: 10px;">
            <button class="try-it" onclick="predict()">Predict Hardness</button>
            <div id="result" class="result"></div>
            
            <h2>📡 API Endpoints</h2>
            <div class="endpoint">
                <strong>POST /predict</strong><br>
                <code>{"composition": "Al0.5CoCrFeNi", "target_property": "hardness"}</code>
            </div>
            <div class="endpoint">
                <strong>POST /literature</strong><br>
                <code>{"question": "Why is hardness high in HEAs?"}</code>
            </div>
            <div class="endpoint">
                <strong>GET /health</strong><br>
                Check model health
            </div>
            <div class="endpoint">
                <strong>POST /batch_predict</strong><br>
                Batch prediction for multiple alloys
            </div>
            
            <h2>📚 Interactive Docs</h2>
            <p><a href="/docs" target="_blank">Swagger UI</a> | <a href="/redoc" target="_blank">ReDoc</a></p>
        </div>
        
        <script>
        async function predict() {
            const comp = document.getElementById('composition').value;
            const resultDiv = document.getElementById('result');
            resultDiv.style.display = 'block';
            resultDiv.innerHTML = 'Predicting...';
            
            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({composition: comp, target_property: 'hardness'})
                });
                const data = await response.json();
                resultDiv.innerHTML = `
                    <strong>Predicted Hardness:</strong> ${data.predicted_value} HV<br>
                    <strong>Uncertainty:</strong> ${data.uncertainty}<br>
                    <strong>Physics Features:</strong><br>
                    <pre>${JSON.stringify(data.physics_features, null, 2)}</pre>
                `;
            } catch(e) {
                resultDiv.innerHTML = 'Error: ' + e.message;
            }
        }
        </script>
    </body>
    </html>
    """

# ----------------------------------------------------------------------
# API Endpoints
# ----------------------------------------------------------------------
@app.post("/predict")
async def predict_alloy(request: PredictionRequest):
    """Predict a property for a given alloy composition."""
    start_time = time.time()
    try:
        feats = compute_physics_features(request.composition)
        X = pd.DataFrame([feats])
        val, unc = pred_agent.predict(X, request.target_property)
        
        PRED_COUNT.inc()
        PRED_LATENCY.observe(time.time() - start_time)
        
        return {
            "composition": request.composition,
            "predicted_value": float(val[0]),
            "property": request.target_property,
            "uncertainty": float(unc),
            "physics_features": feats,
            "success": True
        }
    except Exception as e:
        ERROR_COUNT.inc()
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/explain")
async def explain_prediction(request: PredictionRequest):
    """Get feature importance explanation."""
    try:
        importance = pred_agent.get_feature_importance(request.target_property)
        if not importance:
            raise HTTPException(status_code=404, detail="No importance data available")
        
        top_features = sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
        return {
            "composition": request.composition,
            "target_property": request.target_property,
            "feature_importance": importance,
            "top_features": [{"feature": f[0], "importance": f[1]} for f in top_features]
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/literature")
async def ask_literature(request: LiteratureRequest):
    """Ask a question to the literature intelligence agent."""
    try:
        answer = lit_agent.query(request.question)
        return {"question": request.question, "answer": answer}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/batch_predict")
async def batch_predict(request: BatchRequest):
    """Predict properties for multiple alloys."""
    try:
        results = []
        for comp in request.compositions:
            feats = compute_physics_features(comp)
            X = pd.DataFrame([feats])
            val, unc = pred_agent.predict(X, request.target_property)
            results.append({
                "composition": comp,
                "predicted_value": float(val[0]),
                "uncertainty": float(unc)
            })
        return {"results": results, "target_property": request.target_property}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/health")
async def health_check():
    """Check model health and deployment readiness."""
    try:
        df = load_real_dataset()
        test_df = df.head(5)
        X_test = pd.DataFrame([compute_physics_features(c) for c in test_df["composition"]])
        y_true = {
            "hardness": test_df["hardness_hv"].values,
            "modulus": test_df["modulus_gpa"].values,
            "phase": test_df["phase"].values
        }
        
        metrics = obs_agent.evaluate(X_test, y_true)
        importance_ok = bool(pred_agent.get_feature_importance("hardness"))
        deployable, reason = cicd_agent.evaluate(metrics, importance_ok)
        
        return {
            "status": "healthy" if deployable else "degraded",
            "metrics": metrics,
            "deployable": deployable,
            "reason": reason
        }
    except Exception as e:
        return {"status": "error", "error": str(e), "deployable": False}

@app.post("/retrain")
async def retrain_model(background_tasks: BackgroundTasks):
    """Trigger model retraining."""
    background_tasks.add_task(lambda: pred_agent.train(load_real_dataset()))
    return {"status": "retraining_started", "message": "Model retraining in background"}

@app.get("/metrics")
async def metrics():
    """Prometheus metrics endpoint."""
    return generate_latest(REGISTRY)

@app.get("/info")
async def platform_info():
    """Get platform information."""
    return {
        "name": "Agentic Materials AI Platform",
        "version": "1.0.0",
        "models_available": list(pred_agent.models.keys()) if pred_agent else [],
        "trained": pred_agent.trained if pred_agent else False,
        "feature_names": pred_agent.feature_names if pred_agent else []
    }

# ----------------------------------------------------------------------
# Main Entry Point - Run directly with Python
# ----------------------------------------------------------------------
if __name__ == "__main__":
    import uvicorn
    print("=" * 60)
    print("🧪 Agentic Materials AI Platform")
    print("=" * 60)
    print(f"📍 Server starting at http://localhost:{PORT}")
    print(f"📚 API Docs: http://localhost:{PORT}/docs")
    print(f"📊 Metrics: http://localhost:{PORT}/metrics")
    print("=" * 60)
    print("\n✅ Ready! Press Ctrl+C to stop\n")
    
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

C:\Users\GOWREESWARI\AppData\Local\Temp\ipykernel_14936\2990474839.py:304: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
C:\Users\GOWREESWARI\AppData\Local\Temp\ipykernel_14936\2990474839.py:317: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


🧪 Agentic Materials AI Platform
📍 Server starting at http://localhost:8027
📚 API Docs: http://localhost:8027/docs
📊 Metrics: http://localhost:8027/metrics

✅ Ready! Press Ctrl+C to stop



RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
# Agentic Materials AI Platform - Jupyter Web API (No asyncio)
import os
import json
import time
import re
import random
import threading
import warnings
from typing import Dict, List, Any, Tuple
from io import StringIO

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
from sklearn.inspection import permutation_importance

# Suppress warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
print("=" * 60)
print("🧪 Agentic Materials AI Platform - Starting...")
print("=" * 60)

# ----------------------------------------------------------------------
# Dataset (real HEAs)
# ----------------------------------------------------------------------
REAL_DATASET_CSV = """
composition,hardness_hv,modulus_gpa,phase
Al0.5CoCrFeNi,485,200,FCC+BCC
CoCrFeNi,350,180,FCC
AlCoCrFeNi,520,210,BCC
CrFeNi,280,160,FCC
AlCrFeNi,460,190,BCC
Al0.3CoCrFeNi,420,195,FCC+BCC
Al0.7CoCrFeNi,500,205,BCC
CoCrNi,320,170,FCC
FeNiCoCr,360,185,FCC
Al0.5CrFeNi,440,188,BCC
Al0.5CoCrNi,390,192,FCC+BCC
CoFeNi,300,165,FCC
AlCoCrNi,480,200,BCC
CrFeCoNi,355,182,FCC
AlCrFeCoNi,510,215,BCC
"""

def load_real_dataset() -> pd.DataFrame:
    return pd.read_csv(StringIO(REAL_DATASET_CSV.strip()))

# ----------------------------------------------------------------------
# Physics features
# ----------------------------------------------------------------------
ELEMENT_PROPS = {
    "Al": {"VEC": 3, "r": 143, "EN": 1.61, "Tm": 933, "density": 2.70},
    "Co": {"VEC": 9, "r": 125, "EN": 1.88, "Tm": 1768, "density": 8.90},
    "Cr": {"VEC": 6, "r": 128, "EN": 1.66, "Tm": 2180, "density": 7.19},
    "Fe": {"VEC": 8, "r": 126, "EN": 1.83, "Tm": 1811, "density": 7.87},
    "Ni": {"VEC": 10, "r": 124, "EN": 1.91, "Tm": 1728, "density": 8.91},
    "Cu": {"VEC": 11, "r": 128, "EN": 1.90, "Tm": 1358, "density": 8.96},
    "Mn": {"VEC": 7, "r": 127, "EN": 1.55, "Tm": 1519, "density": 7.47},
    "Ti": {"VEC": 4, "r": 147, "EN": 1.54, "Tm": 1941, "density": 4.51},
    "V":  {"VEC": 5, "r": 134, "EN": 1.63, "Tm": 2183, "density": 6.11},
}

def parse_composition(comp_str: str) -> Dict[str, float]:
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, comp_str)
    comp = {}
    for elem, val in matches:
        comp[elem] = float(val) if val else 1.0
    total = sum(comp.values())
    if total > 0:
        comp = {k: v/total for k,v in comp.items()}
    return comp

def compute_physics_features(composition: str) -> Dict[str, float]:
    comp_frac = parse_composition(composition)
    elements = list(comp_frac.keys())
    fractions = np.array(list(comp_frac.values()))
    
    vec = sum(comp_frac[e] * ELEMENT_PROPS[e]["VEC"] for e in elements if e in ELEMENT_PROPS)
    r_vals = [ELEMENT_PROPS[e]["r"] for e in elements]
    r_bar = sum(fractions * np.array(r_vals))
    delta = np.sqrt(sum(comp_frac[e] * (1 - ELEMENT_PROPS[e]["r"]/r_bar)**2 for e in elements))
    smix = -8.314 * sum(comp_frac[e] * np.log(comp_frac[e]) for e in elements if comp_frac[e]>0)
    hmix = -15.0
    tm_vals = [ELEMENT_PROPS[e]["Tm"] for e in elements]
    tm_avg = sum(fractions * np.array(tm_vals))
    omega = tm_avg * smix / abs(hmix) if hmix != 0 else 0
    en_vals = [ELEMENT_PROPS[e]["EN"] for e in elements]
    en_diff = np.std(en_vals) if len(en_vals)>1 else 0
    dens = sum(comp_frac[e] * ELEMENT_PROPS[e]["density"] for e in elements)
    
    return {
        "VEC": vec,
        "atomic_size_mismatch_delta": delta,
        "mixing_entropy_Smix": smix,
        "mixing_enthalpy_Hmix": hmix,
        "Omega": omega,
        "electronegativity_diff": en_diff,
        "density_g_cm3": dens,
        "melting_point_K": tm_avg
    }

# ----------------------------------------------------------------------
# Prediction Agent
# ----------------------------------------------------------------------
class PredictionAgent:
    def __init__(self):
        self.models = {}
        self.trained = False
        self.X_train = None
        self.y_train = {}
        self.feature_names = None

    def train(self, df: pd.DataFrame):
        features_list = [compute_physics_features(comp) for comp in df["composition"]]
        X = pd.DataFrame(features_list)
        self.X_train = X
        self.feature_names = list(X.columns)
        
        self.y_train["hardness"] = df["hardness_hv"].values
        self.y_train["modulus"] = df["modulus_gpa"].values
        self.y_train["phase"] = df["phase"].values
        
        self.models["hardness"] = RandomForestRegressor(n_estimators=100, random_state=42)
        self.models["hardness"].fit(X, self.y_train["hardness"])
        
        self.models["modulus"] = RandomForestRegressor(n_estimators=100, random_state=42)
        self.models["modulus"].fit(X, self.y_train["modulus"])
        
        self.models["phase"] = RandomForestClassifier(n_estimators=100, random_state=42)
        self.models["phase"].fit(X, self.y_train["phase"])
        
        self.trained = True
        print(f"✅ Trained - Hardness R²: {self.models['hardness'].score(X, self.y_train['hardness']):.3f}")
        print(f"✅ Trained - Modulus R²: {self.models['modulus'].score(X, self.y_train['modulus']):.3f}")
        print(f"✅ Trained - Phase Acc: {self.models['phase'].score(X, self.y_train['phase']):.3f}")

    def predict(self, features: pd.DataFrame, prop: str) -> Tuple[np.ndarray, float]:
        if not self.trained:
            raise RuntimeError("Model not trained")
        model = self.models[prop]
        preds = model.predict(features)
        
        if prop in ["hardness", "modulus"]:
            tree_preds = np.array([tree.predict(features.values) for tree in model.estimators_])
            unc = np.std(tree_preds, axis=0).mean()
        else:
            proba = model.predict_proba(features.values)
            unc = -np.sum(proba * np.log(proba + 1e-9), axis=1).mean()
        return preds, unc

    def get_feature_importance(self, prop: str) -> Dict[str, float]:
        if prop not in self.models or self.X_train is None:
            return {}
        
        model = self.models[prop]
        try:
            if prop in ["hardness", "modulus"]:
                result = permutation_importance(
                    model, self.X_train, self.y_train[prop],
                    n_repeats=5, random_state=42, scoring='r2'
                )
            else:
                result = permutation_importance(
                    model, self.X_train, self.y_train[prop],
                    n_repeats=5, random_state=42, scoring='accuracy'
                )
            return {self.feature_names[i]: float(result.importances_mean[i]) 
                   for i in range(len(self.feature_names))}
        except:
            if hasattr(model, 'feature_importances_'):
                return {self.feature_names[i]: float(model.feature_importances_[i]) 
                       for i in range(len(self.feature_names))}
            return {}

# ----------------------------------------------------------------------
# Literature Agent
# ----------------------------------------------------------------------
class LiteratureAgent:
    def __init__(self):
        self.knowledge_base = {
            "hardness": "High hardness in HEAs comes from severe lattice distortion, solid solution strengthening, and precipitation of secondary phases.",
            "bcc": "BCC formation is favored by high VEC values (>8) and large atomic size mismatch.",
            "fcc": "FCC phases are stabilized by lower VEC values and elements like Co, Cr, Fe, Ni.",
            "vec": "Valence Electron Concentration (VEC) is a key parameter for predicting phase stability.",
            "strength": "Strength increases with Al addition due to lattice distortion and intermetallic formation.",
            "ductility": "Ductility is generally higher in FCC-dominated HEAs compared to BCC-dominated ones.",
            "entropy": "High mixing entropy promotes solid solution formation over intermetallic compounds."
        }
    
    def query(self, question: str) -> str:
        question_lower = question.lower()
        for key, answer in self.knowledge_base.items():
            if key in question_lower:
                return answer
        return "Based on literature: " + list(self.knowledge_base.values())[0]

# ----------------------------------------------------------------------
# Observability Agents
# ----------------------------------------------------------------------
class ObservabilityAgent:
    def __init__(self, agent: PredictionAgent):
        self.agent = agent
        self.history = []
    
    def evaluate(self, X_test: pd.DataFrame, y_true: Dict[str, np.ndarray]) -> Dict:
        report = {}
        for prop, model in self.agent.models.items():
            y_pred = model.predict(X_test)
            if prop in ["hardness", "modulus"]:
                r2 = r2_score(y_true[prop], y_pred)
                mae = mean_absolute_error(y_true[prop], y_pred)
                report[prop] = {"R2": round(r2, 3), "MAE": round(mae, 2), 
                               "status": "good" if r2 > 0.7 else "warning"}
            else:
                acc = accuracy_score(y_true[prop], y_pred)
                report[prop] = {"accuracy": round(acc, 3), 
                               "status": "good" if acc > 0.8 else "warning"}
        
        report["missing_values"] = int(X_test.isnull().sum().sum())
        _, unc = self.agent.predict(X_test.iloc[:1], "hardness")
        report["avg_uncertainty"] = round(float(unc), 2)
        self.history.append(report)
        return report

class CICDGateAgent:
    def evaluate(self, metrics: Dict, importance_ok: bool) -> Tuple[bool, str]:
        models_ok = all(m.get("status") == "good" for m in metrics.values() if isinstance(m, dict))
        unc_ok = metrics.get("avg_uncertainty", 0) < 25.0
        if models_ok and unc_ok and importance_ok:
            return True, "Model meets all quality standards"
        return False, "Model does not meet deployment criteria"

# ----------------------------------------------------------------------
# Initialize Agents
# ----------------------------------------------------------------------
print("\n📊 Training models on HEA dataset...")
pred_agent = PredictionAgent()
pred_agent.train(load_real_dataset())

print("\n📚 Initializing Literature Agent...")
lit_agent = LiteratureAgent()

print("\n🔍 Initializing Observability Agent...")
obs_agent = ObservabilityAgent(pred_agent)

print("\n🚦 Initializing CI/CD Gate Agent...")
cicd_agent = CICDGateAgent()

print("\n" + "=" * 60)
print("✅ All agents initialized successfully!")
print("=" * 60)

# ----------------------------------------------------------------------
# API Functions (No asyncio, synchronous)
# ----------------------------------------------------------------------

def predict_alloy(composition: str, target_property: str = "hardness") -> Dict:
    """Predict property for an alloy composition."""
    try:
        start_time = time.time()
        feats = compute_physics_features(composition)
        X = pd.DataFrame([feats])
        val, unc = pred_agent.predict(X, target_property)
        
        return {
            "composition": composition,
            "predicted_value": float(val[0]),
            "property": target_property,
            "uncertainty": float(unc),
            "physics_features": feats,
            "inference_time_ms": round((time.time() - start_time) * 1000, 2),
            "success": True
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

def explain_prediction(composition: str, target_property: str = "hardness") -> Dict:
    """Get feature importance explanation."""
    try:
        importance = pred_agent.get_feature_importance(target_property)
        if not importance:
            return {"success": False, "error": "No importance data available"}
        
        top_features = sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
        return {
            "composition": composition,
            "target_property": target_property,
            "feature_importance": importance,
            "top_features": [{"feature": f[0], "importance": f[1]} for f in top_features],
            "success": True
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

def ask_literature(question: str) -> Dict:
    """Ask a question to the literature intelligence agent."""
    try:
        answer = lit_agent.query(question)
        return {"question": question, "answer": answer, "success": True}
    except Exception as e:
        return {"success": False, "error": str(e)}

def check_model_health() -> Dict:
    """Check model health and deployment readiness."""
    try:
        df = load_real_dataset()
        test_df = df.head(5)
        X_test = pd.DataFrame([compute_physics_features(c) for c in test_df["composition"]])
        y_true = {
            "hardness": test_df["hardness_hv"].values,
            "modulus": test_df["modulus_gpa"].values,
            "phase": test_df["phase"].values
        }
        
        metrics = obs_agent.evaluate(X_test, y_true)
        importance_ok = bool(pred_agent.get_feature_importance("hardness"))
        deployable, reason = cicd_agent.evaluate(metrics, importance_ok)
        
        return {
            "status": "healthy" if deployable else "degraded",
            "metrics": metrics,
            "deployable": deployable,
            "reason": reason,
            "success": True
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

def batch_predict(compositions: List[str], target_property: str = "hardness") -> Dict:
    """Predict properties for multiple alloys."""
    try:
        results = []
        for comp in compositions:
            feats = compute_physics_features(comp)
            X = pd.DataFrame([feats])
            val, unc = pred_agent.predict(X, target_property)
            results.append({
                "composition": comp,
                "predicted_value": float(val[0]),
                "uncertainty": float(unc)
            })
        return {"results": results, "target_property": target_property, "success": True}
    except Exception as e:
        return {"success": False, "error": str(e)}

def retrain_model() -> Dict:
    """Retrain the model on the full dataset."""
    try:
        print("🔄 Retraining model...")
        pred_agent.train(load_real_dataset())
        return {"status": "retrained", "message": "Model retrained successfully", "success": True}
    except Exception as e:
        return {"success": False, "error": str(e)}

# ----------------------------------------------------------------------
# Print usage examples
# ----------------------------------------------------------------------
print("\n" + "=" * 60)
print("🎯 Agentic Materials AI Platform - Ready for use!")
print("=" * 60)
print("\n📋 Available Functions:")
print("  - predict_alloy(composition, target_property)")
print("  - explain_prediction(composition, target_property)")
print("  - ask_literature(question)")
print("  - check_model_health()")
print("  - batch_predict(compositions, target_property)")
print("  - retrain_model()")
print("\n" + "=" * 60)

# ----------------------------------------------------------------------
# Demo
# ----------------------------------------------------------------------
print("\n🔬 Quick Demo:\n")

# Test prediction
print("📊 Prediction for Al0.5CoCrFeNi:")
result = predict_alloy("Al0.5CoCrFeNi")
if result.get("success"):
    print(f"   Hardness: {result['predicted_value']:.1f} HV")
    print(f"   Uncertainty: {result['uncertainty']:.2f}")
    print(f"   Inference time: {result['inference_time_ms']} ms")
else:
    print(f"   Error: {result.get('error')}")

print("\n📚 Literature Q&A:")
answer = ask_literature("Why is hardness high in HEAs?")
if answer.get("success"):
    print(f"   Q: Why is hardness high in HEAs?")
    print(f"   A: {answer['answer']}")
else:
    print(f"   Error: {answer.get('error')}")

print("\n🏥 Model Health:")
health = check_model_health()
if health.get("success"):
    for prop, metrics in health['metrics'].items():
        if isinstance(metrics, dict):
            print(f"   {prop}: {metrics}")
    print(f"   Deployable: {health['deployable']}")
    print(f"   Reason: {health['reason']}")
else:
    print(f"   Error: {health.get('error')}")

print("\n" + "=" * 60)
print("✨ Platform ready! Try your own queries:")
print("=" * 60)

🧪 Agentic Materials AI Platform - Starting...

📊 Training models on HEA dataset...
✅ Trained - Hardness R²: 0.970
✅ Trained - Modulus R²: 0.963
✅ Trained - Phase Acc: 1.000

📚 Initializing Literature Agent...

🔍 Initializing Observability Agent...

🚦 Initializing CI/CD Gate Agent...

✅ All agents initialized successfully!

🎯 Agentic Materials AI Platform - Ready for use!

📋 Available Functions:
  - predict_alloy(composition, target_property)
  - explain_prediction(composition, target_property)
  - ask_literature(question)
  - check_model_health()
  - batch_predict(compositions, target_property)
  - retrain_model()


🔬 Quick Demo:

📊 Prediction for Al0.5CoCrFeNi:
   Hardness: 457.8 HV
   Uncertainty: 46.67
   Inference time: 34.53 ms

📚 Literature Q&A:
   Q: Why is hardness high in HEAs?
   A: High hardness in HEAs comes from severe lattice distortion, solid solution strengthening, and precipitation of secondary phases.

🏥 Model Health:
   hardness: {'R2': 0.961, 'MAE': 13.7, 'status': 

In [9]:
#!/usr/bin/env python3
"""
================================================================================
AGENTIC MATERIALS AI PLATFORM - COMPLETE PRODUCTION CODE
================================================================================
Features:
- Autonomous Materials Discovery for HEAs/RHEAs
- Literature Intelligence with RAG
- Physics-Informed ML Predictions (Hardness, Modulus, Phase)
- Model Observability & Auto-Retraining
- CI/CD Quality Gates
- REST API with Swagger Docs
- Works in Jupyter Notebook AND as standalone web server
================================================================================
"""

import os
import json
import re
import time
import random
import warnings
import threading
from typing import Dict, List, Any, Tuple, Optional
from io import StringIO
from dataclasses import dataclass, field
from contextlib import contextmanager

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Data science libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

# Web framework - check if running in Jupyter
try:
    from IPython import get_ipython
    IN_JUPYTER = get_ipython() is not None
except:
    IN_JUPYTER = False

if IN_JUPYTER:
    # For Jupyter - use thread-based server
    from http.server import HTTPServer, BaseHTTPRequestHandler
    import urllib.parse
else:
    # For standalone - use FastAPI
    from fastapi import FastAPI, HTTPException, BackgroundTasks
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import HTMLResponse, JSONResponse
    from pydantic import BaseModel, Field
    import uvicorn

# =================================================================================
# PART 1: DATASET & MATERIALS KNOWLEDGE BASE
# =================================================================================

# Real HEA dataset (15 alloys with measured properties)
HEA_DATASET_CSV = """
composition,hardness_hv,modulus_gpa,phase,yield_strength_mpa
Al0.5CoCrFeNi,485,200,FCC+BCC,850
CoCrFeNi,350,180,FCC,550
AlCoCrFeNi,520,210,BCC,1100
CrFeNi,280,160,FCC,400
AlCrFeNi,460,190,BCC,780
Al0.3CoCrFeNi,420,195,FCC+BCC,720
Al0.7CoCrFeNi,500,205,BCC,980
CoCrNi,320,170,FCC,520
FeNiCoCr,360,185,FCC,580
Al0.5CrFeNi,440,188,BCC,750
Al0.5CoCrNi,390,192,FCC+BCC,680
CoFeNi,300,165,FCC,450
AlCoCrNi,480,200,BCC,820
CrFeCoNi,355,182,FCC,560
AlCrFeCoNi,510,215,BCC,1050
"""

# Elemental properties for physics-informed features
ELEMENT_PROPERTIES = {
    "Al": {"VEC": 3, "atomic_radius": 143, "electronegativity": 1.61, "melting_point": 933, "density": 2.70, "mass": 26.98},
    "Co": {"VEC": 9, "atomic_radius": 125, "electronegativity": 1.88, "melting_point": 1768, "density": 8.90, "mass": 58.93},
    "Cr": {"VEC": 6, "atomic_radius": 128, "electronegativity": 1.66, "melting_point": 2180, "density": 7.19, "mass": 52.00},
    "Fe": {"VEC": 8, "atomic_radius": 126, "electronegativity": 1.83, "melting_point": 1811, "density": 7.87, "mass": 55.85},
    "Ni": {"VEC": 10, "atomic_radius": 124, "electronegativity": 1.91, "melting_point": 1728, "density": 8.91, "mass": 58.69},
    "Cu": {"VEC": 11, "atomic_radius": 128, "electronegativity": 1.90, "melting_point": 1358, "density": 8.96, "mass": 63.55},
    "Mn": {"VEC": 7, "atomic_radius": 127, "electronegativity": 1.55, "melting_point": 1519, "density": 7.47, "mass": 54.94},
    "Ti": {"VEC": 4, "atomic_radius": 147, "electronegativity": 1.54, "melting_point": 1941, "density": 4.51, "mass": 47.87},
    "V":  {"VEC": 5, "atomic_radius": 134, "electronegativity": 1.63, "melting_point": 2183, "density": 6.11, "mass": 50.94},
    "Mo": {"VEC": 6, "atomic_radius": 139, "electronegativity": 2.16, "melting_point": 2896, "density": 10.28, "mass": 95.95},
    "W":  {"VEC": 6, "atomic_radius": 139, "electronegativity": 2.36, "melting_point": 3695, "density": 19.25, "mass": 183.84},
    "Nb": {"VEC": 5, "atomic_radius": 146, "electronegativity": 1.60, "melting_point": 2750, "density": 8.57, "mass": 92.91},
    "Zr": {"VEC": 4, "atomic_radius": 160, "electronegativity": 1.33, "melting_point": 2128, "density": 6.52, "mass": 91.22},
}

# Literature knowledge base for RAG
LITERATURE_KNOWLEDGE = {
    "hardness": "High hardness in HEAs results from severe lattice distortion, solid solution strengthening, and precipitation of secondary phases like BCC.",
    "bcc": "Body-Centered Cubic (BCC) phases are favored by high VEC values (>8), large atomic size mismatch, and elements like Al, Cr, Fe.",
    "fcc": "Face-Centered Cubic (FCC) phases are stabilized by lower VEC values (<8) and elements like Co, Cr, Fe, Ni, Mn.",
    "vec": "Valence Electron Concentration (VEC) is a critical parameter for predicting phase stability: VEC > 8 → BCC, VEC < 8 → FCC.",
    "strength": "Yield strength increases with Al addition due to lattice distortion and formation of intermetallic compounds.",
    "ductility": "Ductility is generally higher in FCC-dominated HEAs compared to BCC-dominated ones.",
    "entropy": "High mixing entropy (ΔS_mix) promotes solid solution formation over intermetallic compounds.",
    "temperature": "Higher temperatures stabilize FCC phases, while lower temperatures promote BCC formation.",
    "processing": "Thermomechanical processing (rolling, annealing) can refine grains and improve mechanical properties.",
    "healing": "Self-healing in HEAs can occur through precipitation of secondary phases at crack tips.",
    "corrosion": "HEAs with high Cr and Mo content show excellent corrosion resistance in aggressive environments.",
}


# =================================================================================
# PART 2: PHYSICS-INFORMED FEATURE ENGINEERING
# =================================================================================

def parse_composition(composition: str) -> Dict[str, float]:
    """
    Parse alloy composition string into atomic fractions.
    Example: "Al0.5CoCrFeNi" -> {'Al': 0.5, 'Co': 1.0, 'Cr': 1.0, 'Fe': 1.0, 'Ni': 1.0}
    Then normalizes to sum = 1.
    """
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, composition)
    
    composition_dict = {}
    for element, value in matches:
        composition_dict[element] = float(value) if value else 1.0
    
    # Normalize to sum = 1 (atomic fractions)
    total = sum(composition_dict.values())
    if total > 0:
        composition_dict = {k: v/total for k, v in composition_dict.items()}
    
    return composition_dict


def compute_physics_features(composition: str) -> Dict[str, float]:
    """
    Compute physics-informed features for alloy composition.
    Features: VEC, atomic size mismatch, mixing entropy, enthalpy, Omega, etc.
    """
    comp_frac = parse_composition(composition)
    elements = list(comp_frac.keys())
    fractions = np.array(list(comp_frac.values()))
    
    # 1. Valence Electron Concentration (VEC)
    vec = sum(comp_frac[e] * ELEMENT_PROPERTIES[e]["VEC"] 
              for e in elements if e in ELEMENT_PROPERTIES)
    
    # 2. Atomic size mismatch (δ)
    atomic_radii = [ELEMENT_PROPERTIES[e]["atomic_radius"] for e in elements if e in ELEMENT_PROPERTIES]
    if len(atomic_radii) > 0:
        mean_radius = sum(fractions[:len(atomic_radii)] * np.array(atomic_radii))
        delta = np.sqrt(sum(comp_frac[e] * (1 - ELEMENT_PROPERTIES[e]["atomic_radius"]/mean_radius)**2 
                           for e in elements if e in ELEMENT_PROPERTIES))
    else:
        delta = 0
    
    # 3. Mixing entropy (ΔS_mix) - J/(mol·K)
    smix = -8.314 * sum(comp_frac[e] * np.log(comp_frac[e]) 
                        for e in elements if e in ELEMENT_PROPERTIES and comp_frac[e] > 0)
    
    # 4. Mixing enthalpy (ΔH_mix) - simplified (would use Miedema model in production)
    hmix = -12.0 if len(elements) > 2 else -8.0
    
    # 5. Omega parameter (phase stability)
    melting_points = [ELEMENT_PROPERTIES[e]["melting_point"] for e in elements if e in ELEMENT_PROPERTIES]
    if len(melting_points) > 0:
        tm_avg = sum(fractions[:len(melting_points)] * np.array(melting_points))
        omega = tm_avg * smix / abs(hmix) if hmix != 0 else 0
    else:
        omega = 0
    
    # 6. Electronegativity difference
    en_values = [ELEMENT_PROPERTIES[e]["electronegativity"] for e in elements if e in ELEMENT_PROPERTIES]
    en_diff = np.std(en_values) if len(en_values) > 1 else 0
    
    # 7. Average density
    densities = [ELEMENT_PROPERTIES[e]["density"] for e in elements if e in ELEMENT_PROPERTIES]
    density = sum(fractions[:len(densities)] * np.array(densities)) if densities else 0
    
    # 8. Average melting point
    tm = tm_avg if 'tm_avg' in locals() else 1500
    
    return {
        "VEC": round(vec, 3),
        "atomic_size_mismatch_delta": round(delta, 4),
        "mixing_entropy_Smix": round(smix, 2),
        "mixing_enthalpy_Hmix": hmix,
        "Omega_parameter": round(omega, 2),
        "electronegativity_diff": round(en_diff, 3),
        "density_g_cm3": round(density, 2),
        "melting_point_K": round(tm, 0),
        "num_elements": len(elements)
    }


# =================================================================================
# PART 3: MACHINE LEARNING MODEL AGENT
# =================================================================================

class MLModelAgent:
    """Machine Learning agent for alloy property prediction"""
    
    def __init__(self):
        self.models = {}
        self.label_encoders = {}
        self.trained = False
        self.X_train = None
        self.y_train = {}
        self.feature_names = None
        self.training_history = []
        
    def load_and_train(self, dataset_csv: str = HEA_DATASET_CSV):
        """Load dataset and train all models"""
        print("📊 Loading HEA dataset...")
        df = pd.read_csv(StringIO(dataset_csv.strip()))
        
        # Compute features for all compositions
        print("🔬 Computing physics-informed features...")
        features_list = []
        for comp in df["composition"]:
            features_list.append(compute_physics_features(comp))
        
        X = pd.DataFrame(features_list)
        self.X_train = X
        self.feature_names = list(X.columns)
        
        # Store training data
        self.y_train["hardness"] = df["hardness_hv"].values
        self.y_train["modulus"] = df["modulus_gpa"].values
        
        # Encode phase labels
        le = LabelEncoder()
        self.y_train["phase_encoded"] = le.fit_transform(df["phase"].values)
        self.label_encoders["phase"] = le
        
        # Train Random Forest models
        print("🤖 Training ML models...")
        
        # Hardness Regressor
        self.models["hardness"] = RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        self.models["hardness"].fit(X, self.y_train["hardness"])
        
        # Modulus Regressor
        self.models["modulus"] = RandomForestRegressor(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        self.models["modulus"].fit(X, self.y_train["modulus"])
        
        # Phase Classifier
        self.models["phase"] = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        self.models["phase"].fit(X, self.y_train["phase_encoded"])
        
        self.trained = True
        
        # Print performance metrics
        hardness_r2 = self.models["hardness"].score(X, self.y_train["hardness"])
        modulus_r2 = self.models["modulus"].score(X, self.y_train["modulus"])
        phase_acc = self.models["phase"].score(X, self.y_train["phase_encoded"])
        
        print(f"✅ Training complete!")
        print(f"   - Hardness R²: {hardness_r2:.3f}")
        print(f"   - Modulus R²: {modulus_r2:.3f}")
        print(f"   - Phase Accuracy: {phase_acc:.3f}")
        
        return {"hardness_r2": hardness_r2, "modulus_r2": modulus_r2, "phase_acc": phase_acc}
    
    def predict(self, composition: str, property_name: str = "hardness") -> Tuple[float, float]:
        """Predict property and return (value, uncertainty)"""
        if not self.trained:
            raise RuntimeError("Model not trained. Call load_and_train() first.")
        
        features = compute_physics_features(composition)
        X_pred = pd.DataFrame([features])
        
        model = self.models.get(property_name)
        if not model:
            raise ValueError(f"Unknown property: {property_name}")
        
        prediction = model.predict(X_pred)[0]
        
        # Calculate uncertainty
        if property_name in ["hardness", "modulus"]:
            # Standard deviation of tree predictions for regression
            tree_preds = np.array([tree.predict(X_pred.values) for tree in model.estimators_])
            uncertainty = np.std(tree_preds).item()
        else:
            # Entropy for classification
            proba = model.predict_proba(X_pred.values)[0]
            uncertainty = -np.sum(proba * np.log(proba + 1e-9)).item()
        
        # Decode phase if needed
        if property_name == "phase" and hasattr(self, 'label_encoders'):
            phase_label = self.label_encoders["phase"].inverse_transform([int(prediction)])[0]
            return phase_label, uncertainty
        
        return float(prediction), uncertainty
    
    def get_feature_importance(self, property_name: str = "hardness") -> Dict[str, float]:
        """Get feature importance using permutation importance"""
        if not self.trained:
            return {}
        
        model = self.models.get(property_name)
        if not model or self.X_train is None:
            return {}
        
        try:
            if property_name in ["hardness", "modulus"]:
                result = permutation_importance(
                    model, self.X_train, self.y_train[property_name],
                    n_repeats=5, random_state=42, scoring='r2'
                )
            else:
                result = permutation_importance(
                    model, self.X_train, self.y_train["phase_encoded"],
                    n_repeats=5, random_state=42, scoring='accuracy'
                )
            
            importance = {self.feature_names[i]: float(result.importances_mean[i]) 
                         for i in range(len(self.feature_names))}
            return dict(sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True))
        except:
            # Fallback to built-in feature importance
            if hasattr(model, 'feature_importances_'):
                importance = {self.feature_names[i]: float(model.feature_importances_[i]) 
                             for i in range(len(self.feature_names))}
                return dict(sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True))
            return {}
    
    def retrain(self):
        """Retrain the model (useful when new data is added)"""
        print("🔄 Retraining models...")
        return self.load_and_train()


# =================================================================================
# PART 4: LITERATURE INTELLIGENCE AGENT (RAG)
# =================================================================================

class LiteratureAgent:
    """RAG agent for answering materials science questions"""
    
    def __init__(self, knowledge_base: Dict = None):
        self.knowledge_base = knowledge_base or LITERATURE_KNOWLEDGE
        self.conversation_history = []
        
    def query(self, question: str) -> str:
        """Answer a question using the literature knowledge base"""
        question_lower = question.lower()
        
        # Keyword matching with context
        for keyword, answer in self.knowledge_base.items():
            if keyword in question_lower:
                self.conversation_history.append({"question": question, "answer": answer})
                return answer
        
        # Default response with most relevant info
        if any(word in question_lower for word in ["what", "explain", "describe"]):
            return "Based on materials science literature: " + list(self.knowledge_base.values())[0]
        
        return "I couldn't find specific information in the literature. Please try rephrasing your question."
    
    def get_conversation_history(self) -> List[Dict]:
        """Get the conversation history"""
        return self.conversation_history


# =================================================================================
# PART 5: OBSERVABILITY & AUTO-RETRAIN AGENT
# =================================================================================

class ObservabilityAgent:
    """Monitors model health and performance"""
    
    def __init__(self, ml_agent: MLModelAgent):
        self.ml_agent = ml_agent
        self.metrics_history = []
        self.alerts = []
        
    def evaluate_model_health(self) -> Dict:
        """Evaluate model health metrics"""
        if not self.ml_agent.trained:
            return {"status": "not_trained", "error": "Model not trained"}
        
        # Use last 20% of training data as validation
        X_test = self.ml_agent.X_train.iloc[-int(len(self.ml_agent.X_train) * 0.2):]
        y_hardness_true = self.ml_agent.y_train["hardness"][-len(X_test):]
        y_modulus_true = self.ml_agent.y_train["modulus"][-len(X_test):]
        
        y_hardness_pred = self.ml_agent.models["hardness"].predict(X_test)
        y_modulus_pred = self.ml_agent.models["modulus"].predict(X_test)
        
        hardness_r2 = r2_score(y_hardness_true, y_hardness_pred)
        hardness_mae = mean_absolute_error(y_hardness_true, y_hardness_pred)
        modulus_r2 = r2_score(y_modulus_true, y_modulus_pred)
        modulus_mae = mean_absolute_error(y_modulus_true, y_modulus_pred)
        
        health_status = {
            "timestamp": time.time(),
            "hardness": {
                "r2_score": round(hardness_r2, 3),
                "mae": round(hardness_mae, 2),
                "status": "good" if hardness_r2 > 0.7 else "warning"
            },
            "modulus": {
                "r2_score": round(modulus_r2, 3),
                "mae": round(modulus_mae, 2),
                "status": "good" if modulus_r2 > 0.7 else "warning"
            },
            "overall_status": "healthy" if (hardness_r2 > 0.7 and modulus_r2 > 0.7) else "degraded"
        }
        
        self.metrics_history.append(health_status)
        
        # Check for alerts
        if hardness_r2 < 0.7:
            self.alerts.append(f"⚠️ Hardness model R² dropped to {hardness_r2:.3f}")
        if modulus_r2 < 0.7:
            self.alerts.append(f"⚠️ Modulus model R² dropped to {modulus_r2:.3f}")
        
        return health_status
    
    def should_retrain(self) -> bool:
        """Check if model should be retrained"""
        health = self.evaluate_model_health()
        
        retrain_conditions = [
            health["hardness"]["r2_score"] < 0.7,
            health["modulus"]["r2_score"] < 0.7,
            len(self.alerts) > 3
        ]
        
        return any(retrain_conditions)


class CICDGateAgent:
    """CI/CD quality gate for model deployment"""
    
    def __init__(self, ml_agent: MLModelAgent):
        self.ml_agent = ml_agent
        
    def evaluate_deployment_readiness(self) -> Tuple[bool, List[str]]:
        """Check if model is ready for deployment"""
        checks = []
        passed = True
        
        # Check 1: Model is trained
        if not self.ml_agent.trained:
            checks.append("❌ Model not trained")
            passed = False
        else:
            checks.append("✅ Model trained")
        
        # Check 2: Feature importance exists
        importance = self.ml_agent.get_feature_importance("hardness")
        if importance:
            checks.append(f"✅ Feature importance available ({len(importance)} features)")
        else:
            checks.append("⚠️ Feature importance not available")
        
        # Check 3: Model performance
        hardness_importance = importance.get("VEC", 0) if importance else 0
        if hardness_importance > 0:
            checks.append(f"✅ Top feature VEC importance: {hardness_importance:.3f}")
        
        # Check 4: API test (simulated)
        try:
            prediction, _ = self.ml_agent.predict("Al0.5CoCrFeNi", "hardness")
            if prediction > 0:
                checks.append(f"✅ API test passed (prediction: {prediction:.0f})")
            else:
                checks.append("❌ API test failed")
                passed = False
        except:
            checks.append("❌ API test failed")
            passed = False
        
        return passed, checks


# =================================================================================
# PART 6: AGENTIC WORKFLOW ORCHESTRATOR
# =================================================================================

class MaterialsAgenticOrchestrator:
    """Main orchestrator for all agents"""
    
    def __init__(self):
        self.ml_agent = MLModelAgent()
        self.literature_agent = LiteratureAgent()
        self.observability_agent = None
        self.cicd_agent = None
        self.initialized = False
        
    def initialize(self):
        """Initialize all agents and train models"""
        print("\n" + "="*60)
        print("🧪 INITIALIZING AGENTIC MATERIALS AI PLATFORM")
        print("="*60)
        
        # Train ML models
        self.ml_agent.load_and_train()
        
        # Initialize observability
        self.observability_agent = ObservabilityAgent(self.ml_agent)
        self.cicd_agent = CICDGateAgent(self.ml_agent)
        
        self.initialized = True
        
        print("\n✅ All agents initialized successfully!")
        
        # Run health check
        health = self.observability_agent.evaluate_model_health()
        print(f"\n📊 Initial Health Status: {health['overall_status'].upper()}")
        
        return self
    
    def predict_property(self, composition: str, property_name: str = "hardness") -> Dict:
        """Predict material property"""
        if not self.initialized:
            self.initialize()
        
        try:
            start_time = time.time()
            value, uncertainty = self.ml_agent.predict(composition, property_name)
            features = compute_physics_features(composition)
            inference_time = (time.time() - start_time) * 1000
            
            return {
                "success": True,
                "composition": composition,
                "property": property_name,
                "predicted_value": value,
                "uncertainty": uncertainty,
                "physics_features": features,
                "inference_time_ms": round(inference_time, 2)
            }
        except Exception as e:
            return {"success": False, "error": str(e)}
    
    def get_explanation(self, composition: str, property_name: str = "hardness") -> Dict:
        """Get feature importance explanation"""
        if not self.initialized:
            self.initialize()
        
        importance = self.ml_agent.get_feature_importance(property_name)
        top_features = list(importance.keys())[:5] if importance else []
        
        return {
            "composition": composition,
            "property": property_name,
            "feature_importance": importance,
            "top_features": top_features,
            "explanation": f"The prediction is primarily driven by {', '.join(top_features[:3])}."
        }
    
    def ask_literature(self, question: str) -> str:
        """Ask literature agent"""
        if not self.initialized:
            self.initialize()
        return self.literature_agent.query(question)
    
    def check_health(self) -> Dict:
        """Check system health"""
        if not self.initialized:
            self.initialize()
        return self.observability_agent.evaluate_model_health()
    
    def deployment_gate(self) -> Dict:
        """Check CI/CD deployment readiness"""
        if not self.initialized:
            self.initialize()
        passed, checks = self.cicd_agent.evaluate_deployment_readiness()
        return {
            "deployable": passed,
            "checks": checks,
            "message": "Ready for deployment" if passed else "Issues found - fix before deployment"
        }
    
    def retrain(self) -> Dict:
        """Force model retraining"""
        if not self.initialized:
            self.initialize()
        result = self.ml_agent.retrain()
        return {"status": "retrained", "metrics": result}
    
    def batch_predict(self, compositions: List[str], property_name: str = "hardness") -> List[Dict]:
        """Batch prediction for multiple compositions"""
        results = []
        for comp in compositions:
            pred = self.predict_property(comp, property_name)
            results.append(pred)
        return results


# =================================================================================
# PART 7: WEB API (FastAPI for standalone, HTTP Server for Jupyter)
# =================================================================================

# Global orchestrator instance
orchestrator = None

def get_orchestrator():
    """Get or create orchestrator instance (singleton)"""
    global orchestrator
    if orchestrator is None:
        orchestrator = MaterialsAgenticOrchestrator()
        orchestrator.initialize()
    return orchestrator


# =================================================================================
# JUPYTER COMPATIBLE HTTP SERVER
# =================================================================================

class JupyterAPIHandler(BaseHTTPRequestHandler):
    """HTTP request handler for Jupyter environment"""
    
    def _send_response(self, status_code: int, data: dict):
        """Send JSON response"""
        self.send_response(status_code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()
        self.wfile.write(json.dumps(data).encode())
    
    def do_OPTIONS(self):
        """Handle CORS preflight"""
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()
    
    def do_GET(self):
        """Handle GET requests"""
        if self.path == '/':
            self._send_response(200, {
                "name": "Agentic Materials AI Platform",
                "version": "2.0.0",
                "status": "running",
                "endpoints": {
                    "predict": "POST /predict",
                    "health": "GET /health",
                    "literature": "POST /literature",
                    "deployment_gate": "GET /deployment_gate",
                    "docs": "GET /docs"
                }
            })
        elif self.path == '/health':
            orch = get_orchestrator()
            health = orch.check_health()
            self._send_response(200, health)
        elif self.path == '/deployment_gate':
            orch = get_orchestrator()
            gate = orch.deployment_gate()
            self._send_response(200, gate)
        elif self.path == '/docs':
            self.send_response(200)
            self.send_header('Content-Type', 'text/html')
            self.end_headers()
            html = self._get_docs_html()
            self.wfile.write(html.encode())
        else:
            self._send_response(404, {"error": "Endpoint not found"})
    
    def do_POST(self):
        """Handle POST requests"""
        content_length = int(self.headers.get('Content-Length', 0))
        post_data = self.rfile.read(content_length)
        
        try:
            data = json.loads(post_data) if content_length > 0 else {}
        except:
            self._send_response(400, {"error": "Invalid JSON"})
            return
        
        if self.path == '/predict':
            orch = get_orchestrator()
            composition = data.get('composition', '')
            property_name = data.get('property', 'hardness')
            
            if not composition:
                self._send_response(400, {"error": "composition is required"})
                return
            
            result = orch.predict_property(composition, property_name)
            self._send_response(200, result)
        
        elif self.path == '/literature':
            orch = get_orchestrator()
            question = data.get('question', '')
            
            if not question:
                self._send_response(400, {"error": "question is required"})
                return
            
            answer = orch.ask_literature(question)
            self._send_response(200, {"question": question, "answer": answer})
        
        elif self.path == '/batch_predict':
            orch = get_orchestrator()
            compositions = data.get('compositions', [])
            property_name = data.get('property', 'hardness')
            
            if not compositions:
                self._send_response(400, {"error": "compositions list is required"})
                return
            
            results = orch.batch_predict(compositions, property_name)
            self._send_response(200, {"results": results})
        
        elif self.path == '/explain':
            orch = get_orchestrator()
            composition = data.get('composition', '')
            property_name = data.get('property', 'hardness')
            
            if not composition:
                self._send_response(400, {"error": "composition is required"})
                return
            
            explanation = orch.get_explanation(composition, property_name)
            self._send_response(200, explanation)
        
        else:
            self._send_response(404, {"error": "Endpoint not found"})
    
    def _get_docs_html(self) -> str:
        """Generate HTML documentation page"""
        return """
        <!DOCTYPE html>
        <html>
        <head>
            <title>Materials AI Platform - API Docs</title>
            <style>
                body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1200px; margin: 0 auto; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }
                .container { background: white; border-radius: 20px; padding: 30px; box-shadow: 0 20px 60px rgba(0,0,0,0.3); }
                h1 { color: #667eea; }
                .endpoint { background: #f0f0f0; padding: 15px; border-radius: 10px; margin: 15px 0; font-family: monospace; }
                .method { display: inline-block; padding: 5px 10px; border-radius: 5px; font-weight: bold; }
                .post { background: #28a745; color: white; }
                .get { background: #007bff; color: white; }
                code { background: #e0e0e0; padding: 2px 5px; border-radius: 3px; }
                pre { background: #2d2d2d; color: #f8f8f2; padding: 15px; border-radius: 10px; overflow-x: auto; }
            </style>
        </head>
        <body>
            <div class="container">
                <h1>🧪 Agentic Materials AI Platform</h1>
                <p>Autonomous Alloy Discovery · Literature Intelligence · ML Predictions · MLOps</p>
                
                <h2>📡 API Endpoints</h2>
                
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/predict</strong><br>
                    Predict alloy property<br>
                    <code>{"composition": "Al0.5CoCrFeNi", "property": "hardness"}</code>
                </div>
                
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/literature</strong><br>
                    Ask literature questions<br>
                    <code>{"question": "Why is hardness high in HEAs?"}</code>
                </div>
                
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/explain</strong><br>
                    Get feature importance explanation<br>
                    <code>{"composition": "Al0.5CoCrFeNi", "property": "hardness"}</code>
                </div>
                
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/batch_predict</strong><br>
                    Batch prediction for multiple alloys<br>
                    <code>{"compositions": ["Al0.5CoCrFeNi", "CoCrFeNi"], "property": "hardness"}</code>
                </div>
                
                <div class="endpoint">
                    <span class="method get">GET</span> <strong>/health</strong><br>
                    Check model health status
                </div>
                
                <div class="endpoint">
                    <span class="method get">GET</span> <strong>/deployment_gate</strong><br>
                    Check CI/CD deployment readiness
                </div>
                
                <h2>🚀 Quick Test</h2>
                <button onclick="testAPI()">Test Prediction</button>
                <pre id="result"></pre>
            </div>
            
            <script>
                async function testAPI() {
                    const response = await fetch('/predict', {
                        method: 'POST',
                        headers: {'Content-Type': 'application/json'},
                        body: JSON.stringify({composition: "Al0.5CoCrFeNi", property: "hardness"})
                    });
                    const data = await response.json();
                    document.getElementById('result').innerHTML = JSON.stringify(data, null, 2);
                }
            </script>
        </body>
        </html>
        """


def start_jupyter_server(port: int = 5026):
    """Start HTTP server in Jupyter environment"""
    server = HTTPServer(('localhost', port), JupyterAPIHandler)
    print(f"\n🚀 API Server running at http://localhost:{port}")
    print(f"📚 API Docs: http://localhost:{port}/docs")
    print(f"💡 Press Ctrl+C to stop\n")
    
    # Start server in a separate thread
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server


# =================================================================================
# PART 8: MAIN ENTRY POINT
# =================================================================================

def main():
    """Main function to run the platform"""
    print("""
    ╔══════════════════════════════════════════════════════════════╗
    ║     AGENTIC MATERIALS AI PLATFORM - COMPLETE EDITION         ║
    ║                                                              ║
    ║  Features:                                                   ║
    ║  • Autonomous Materials Discovery for HEAs/RHEAs            ║
    ║  • Physics-Informed ML Predictions (Hardness/Modulus/Phase) ║
    ║  • Literature Intelligence with RAG                         ║
    ║  • Model Observability & Auto-Retraining                    ║
    ║  • CI/CD Quality Gates                                      ║
    ║  • REST API + Interactive Docs                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """)
    
    # Initialize orchestrator
    orch = get_orchestrator()
    
    # Run demo predictions
    print("\n" + "="*60)
    print("🔬 DEMO PREDICTIONS")
    print("="*60)
    
    test_alloys = ["Al0.5CoCrFeNi", "CoCrFeNi", "AlCoCrFeNi"]
    for alloy in test_alloys:
        result = orch.predict_property(alloy, "hardness")
        if result["success"]:
            print(f"\n📊 {alloy}:")
            print(f"   Predicted Hardness: {result['predicted_value']:.0f} HV")
            print(f"   Uncertainty: ±{result['uncertainty']:.1f}")
            print(f"   Inference Time: {result['inference_time_ms']} ms")
    
    # Literature example
    print("\n" + "="*60)
    print("📚 LITERATURE INTELLIGENCE")
    print("="*60)
    answer = orch.ask_literature("Why is hardness high in HEAs?")
    print(f"\n❓ Q: Why is hardness high in HEAs?")
    print(f"💡 A: {answer}")
    
    # Health check
    print("\n" + "="*60)
    print("🏥 MODEL HEALTH CHECK")
    print("="*60)
    health = orch.check_health()
    print(f"\nOverall Status: {health['overall_status'].upper()}")
    print(f"Hardness R²: {health['hardness']['r2_score']}")
    print(f"Modulus R²: {health['modulus']['r2_score']}")
    
    # CI/CD Gate
    print("\n" + "="*60)
    print("🚦 CI/CD DEPLOYMENT GATE")
    print("="*60)
    gate = orch.deployment_gate()
    print(f"\nDeployable: {gate['deployable']}")
    for check in gate['checks']:
        print(f"  {check}")
    
    # Start API server based on environment
    print("\n" + "="*60)
    print("🌐 STARTING API SERVER")
    print("="*60)
    
    if IN_JUPYTER:
        print("\n📓 Running in Jupyter Notebook - Starting HTTP server...")
        server = start_jupyter_server(port=5026)
        print("\n✅ Server is running in background!")
        print("\n💡 To test the API, run in a new cell:")
        print("""
        import requests
        response = requests.post('http://localhost:5026/predict', 
                                json={'composition': 'Al0.5CoCrFeNi', 'property': 'hardness'})
        print(response.json())
        """)
        # Keep the server running
        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\n👋 Shutting down...")
            server.shutdown()
    else:
        print("\n💻 Running as standalone - Starting FastAPI server...")
        print("📍 API available at http://localhost:8000")
        print("📚 Interactive docs at http://localhost:8000/docs")
        
        # Create FastAPI app
        app = FastAPI(title="Agentic Materials AI Platform", 
                      description="Autonomous Alloy Discovery with AI Agents",
                      version="2.0.0")
        
        app.add_middleware(
            CORSMiddleware,
            allow_origins=["*"],
            allow_credentials=True,
            allow_methods=["*"],
            allow_headers=["*"],
        )
        
        # Define request/response models
        class PredictRequest(BaseModel):
            composition: str = Field(..., example="Al0.5CoCrFeNi")
            property: str = Field("hardness", example="hardness")
        
        class LiteratureRequest(BaseModel):
            question: str = Field(..., example="Why is hardness high in HEAs?")
        
        class BatchRequest(BaseModel):
            compositions: List[str]
            property: str = "hardness"
        
        @app.get("/")
        async def root():
            return {
                "name": "Agentic Materials AI Platform",
                "version": "2.0.0",
                "status": "running",
                "docs": "/docs"
            }
        
        @app.post("/predict")
        async def predict(request: PredictRequest):
            result = orch.predict_property(request.composition, request.property)
            if not result["success"]:
                raise HTTPException(status_code=400, detail=result.get("error"))
            return result
        
        @app.post("/literature")
        async def literature(request: LiteratureRequest):
            answer = orch.ask_literature(request.question)
            return {"question": request.question, "answer": answer}
        
        @app.post("/explain")
        async def explain(request: PredictRequest):
            return orch.get_explanation(request.composition, request.property)
        
        @app.post("/batch_predict")
        async def batch_predict(request: BatchRequest):
            return {"results": orch.batch_predict(request.compositions, request.property)}
        
        @app.get("/health")
        async def health():
            return orch.check_health()
        
        @app.get("/deployment_gate")
        async def deployment_gate():
            return orch.deployment_gate()
        
        @app.get("/info")
        async def info():
            return {
                "name": "Agentic Materials AI Platform",
                "version": "2.0.0",
                "features": ["VEC", "atomic_mismatch", "mixing_entropy"],
                "available_properties": ["hardness", "modulus", "phase"]
            }
        
        @app.get("/docs_html", response_class=HTMLResponse)
        async def docs_html():
            return JupyterAPIHandler._get_docs_html(None)
        
        # Run the server
        uvicorn.run(app, host="0.0.0.0", port=5026)


# =================================================================================
# RUN THE PLATFORM
# =================================================================================
if __name__ == "__main__":
    main()
else:
    # For Jupyter notebook import - initialize orchestrator
    print("📦 Importing Materials AI Platform...")
    orchestrator = MaterialsAgenticOrchestrator()
    orchestrator.initialize()
    print("\n✅ Platform ready! Use 'orchestrator' to access features")


    ╔══════════════════════════════════════════════════════════════╗
    ║     AGENTIC MATERIALS AI PLATFORM - COMPLETE EDITION         ║
    ║                                                              ║
    ║  Features:                                                   ║
    ║  • Autonomous Materials Discovery for HEAs/RHEAs            ║
    ║  • Physics-Informed ML Predictions (Hardness/Modulus/Phase) ║
    ║  • Literature Intelligence with RAG                         ║
    ║  • Model Observability & Auto-Retraining                    ║
    ║  • CI/CD Quality Gates                                      ║
    ║  • REST API + Interactive Docs                              ║
    ╚══════════════════════════════════════════════════════════════╝
    

🧪 INITIALIZING AGENTIC MATERIALS AI PLATFORM
📊 Loading HEA dataset...
🔬 Computing physics-informed features...
🤖 Training ML models...
✅ Training complete!
   - Hardness R²: 0.970
   - Modulus R²: 0.965
   - Phase Accuracy: 1.000

✅ All age

In [11]:
# =====================================================================
# COMPLETE AGENTIC MATERIALS AI PLATFORM - WEB API
# Copy this ENTIRE cell into Jupyter and run - It will start immediately!
# =====================================================================

import json
import re
import time
import threading
import warnings
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from io import StringIO

warnings.filterwarnings('ignore')

print("="*60)
print("🧪 STARTING AGENTIC MATERIALS AI PLATFORM")
print("="*60)

# =====================================================================
# PART 1: DATASET & KNOWLEDGE BASE
# =====================================================================

# Real HEA dataset
HEA_DATASET = """
composition,hardness_hv,modulus_gpa,phase
Al0.5CoCrFeNi,485,200,FCC+BCC
CoCrFeNi,350,180,FCC
AlCoCrFeNi,520,210,BCC
CrFeNi,280,160,FCC
AlCrFeNi,460,190,BCC
Al0.3CoCrFeNi,420,195,FCC+BCC
Al0.7CoCrFeNi,500,205,BCC
CoCrNi,320,170,FCC
FeNiCoCr,360,185,FCC
Al0.5CrFeNi,440,188,BCC
"""

# Literature knowledge base
LITERATURE_KB = {
    "hardness": "High hardness in HEAs comes from severe lattice distortion and solid solution strengthening.",
    "bcc": "BCC phases are favored by high VEC values (>8) and elements like Al, Cr, Fe.",
    "fcc": "FCC phases are stabilized by lower VEC values (<8) and elements like Co, Cr, Fe, Ni.",
    "vec": "Valence Electron Concentration (VEC) is key for phase prediction: VEC>8 → BCC, VEC<8 → FCC.",
    "strength": "Strength increases with Al addition due to lattice distortion.",
    "entropy": "High mixing entropy promotes solid solution formation."
}

# Element properties for physics features
ELEMENT_PROPS = {
    "Al": {"VEC": 3, "radius": 143, "EN": 1.61, "mass": 27},
    "Co": {"VEC": 9, "radius": 125, "EN": 1.88, "mass": 59},
    "Cr": {"VEC": 6, "radius": 128, "EN": 1.66, "mass": 52},
    "Fe": {"VEC": 8, "radius": 126, "EN": 1.83, "mass": 56},
    "Ni": {"VEC": 10, "radius": 124, "EN": 1.91, "mass": 59},
    "Cu": {"VEC": 11, "radius": 128, "EN": 1.90, "mass": 64},
    "Mn": {"VEC": 7, "radius": 127, "EN": 1.55, "mass": 55},
}

# =====================================================================
# PART 2: FEATURE ENGINEERING
# =====================================================================

def parse_composition(composition: str) -> dict:
    """Parse alloy composition e.g., 'Al0.5CoCrFeNi'"""
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, composition)
    comp = {}
    for elem, val in matches:
        comp[elem] = float(val) if val else 1.0
    total = sum(comp.values())
    if total > 0:
        comp = {k: v/total for k, v in comp.items()}
    return comp

def compute_physics_features(composition: str) -> dict:
    """Compute physics-informed features"""
    comp_frac = parse_composition(composition)
    elements = list(comp_frac.keys())
    fractions = np.array(list(comp_frac.values()))
    
    # VEC (Valence Electron Concentration)
    vec = sum(comp_frac[e] * ELEMENT_PROPS[e]["VEC"] for e in elements if e in ELEMENT_PROPS)
    
    # Atomic size mismatch (δ)
    radii = [ELEMENT_PROPS[e]["radius"] for e in elements if e in ELEMENT_PROPS]
    if radii:
        mean_radius = sum(fractions[:len(radii)] * np.array(radii))
        delta = np.sqrt(sum(comp_frac[e] * (1 - ELEMENT_PROPS[e]["radius"]/mean_radius)**2 
                           for e in elements if e in ELEMENT_PROPS))
    else:
        delta = 0
    
    # Mixing entropy (ΔS_mix)
    smix = -8.314 * sum(comp_frac[e] * np.log(comp_frac[e]) 
                        for e in elements if e in ELEMENT_PROPS and comp_frac[e] > 0)
    
    # Electronegativity difference
    en_values = [ELEMENT_PROPS[e]["EN"] for e in elements if e in ELEMENT_PROPS]
    en_diff = np.std(en_values) if len(en_values) > 1 else 0
    
    return {
        "VEC": round(vec, 3),
        "atomic_mismatch": round(delta, 4),
        "mixing_entropy": round(smix, 2),
        "electronegativity_diff": round(en_diff, 3),
        "num_elements": len(elements)
    }

# =====================================================================
# PART 3: TRAIN MACHINE LEARNING MODELS
# =====================================================================

print("\n📊 Loading and training models...")

# Load dataset
df = pd.read_csv(StringIO(HEA_DATASET.strip()))
print(f"   Loaded {len(df)} alloy compositions")

# Compute features for all alloys
features_list = []
for comp in df["composition"]:
    features_list.append(compute_physics_features(comp))

X = pd.DataFrame(features_list)
y_hardness = df["hardness_hv"].values
y_modulus = df["modulus_gpa"].values

# Train hardness model
hardness_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
hardness_model.fit(X, y_hardness)
hardness_r2 = hardness_model.score(X, y_hardness)

# Train modulus model
modulus_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modulus_model.fit(X, y_modulus)
modulus_r2 = modulus_model.score(X, y_modulus)

# Train phase classifier
le = LabelEncoder()
y_phase_encoded = le.fit_transform(df["phase"].values)
phase_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
phase_model.fit(X, y_phase_encoded)
phase_acc = phase_model.score(X, y_phase_encoded)

print(f"   ✅ Hardness model R²: {hardness_r2:.3f}")
print(f"   ✅ Modulus model R²: {modulus_r2:.3f}")
print(f"   ✅ Phase model accuracy: {phase_acc:.3f}")

# =====================================================================
# PART 4: LITERATURE AGENT
# =====================================================================

class LiteratureAgent:
    def query(self, question: str) -> str:
        question_lower = question.lower()
        for keyword, answer in LITERATURE_KB.items():
            if keyword in question_lower:
                return answer
        return "Based on materials science literature: " + list(LITERATURE_KB.values())[0]

literature_agent = LiteratureAgent()

# =====================================================================
# PART 5: HTTP API SERVER
# =====================================================================

class MaterialsAPIHandler(BaseHTTPRequestHandler):
    """HTTP request handler for Materials AI API"""
    
    def log_message(self, format, *args):
        """Suppress default logging"""
        pass
    
    def _send_json(self, data, status_code=200):
        """Send JSON response with CORS headers"""
        self.send_response(status_code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, PUT, DELETE, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type, Authorization')
        self.end_headers()
        self.wfile.write(json.dumps(data, indent=2).encode())
    
    def do_OPTIONS(self):
        """Handle CORS preflight requests"""
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()
    
    def do_GET(self):
        """Handle GET requests"""
        parsed_path = urlparse(self.path)
        path = parsed_path.path
        
        if path == '/' or path == '/':
            self._send_json({
                "name": "Agentic Materials AI Platform",
                "version": "2.0.0",
                "status": "running",
                "endpoints": {
                    "GET /": "API information",
                    "GET /health": "System health check",
                    "GET /docs": "Interactive documentation",
                    "POST /predict": "Predict alloy properties",
                    "POST /literature": "Ask literature questions",
                    "POST /batch_predict": "Batch predictions"
                }
            })
        
        elif path == '/health':
            self._send_json({
                "status": "healthy",
                "models": ["hardness", "modulus", "phase"],
                "hardness_r2": hardness_r2,
                "modulus_r2": modulus_r2,
                "phase_accuracy": phase_acc,
                "samples_trained": len(df)
            })
        
        elif path == '/docs':
            self.send_response(200)
            self.send_header('Content-Type', 'text/html')
            self.end_headers()
            html = self._get_docs_html()
            self.wfile.write(html.encode())
        
        else:
            self._send_json({"error": "Endpoint not found"}, 404)
    
    def do_POST(self):
        """Handle POST requests"""
        content_length = int(self.headers.get('Content-Length', 0))
        post_data = self.rfile.read(content_length)
        
        try:
            data = json.loads(post_data) if content_length > 0 else {}
        except json.JSONDecodeError:
            self._send_json({"error": "Invalid JSON"}, 400)
            return
        
        parsed_path = urlparse(self.path)
        path = parsed_path.path
        
        if path == '/predict':
            composition = data.get('composition', '')
            property_name = data.get('property', 'hardness')
            
            if not composition:
                self._send_json({"error": "composition is required"}, 400)
                return
            
            try:
                # Compute features
                features = compute_physics_features(composition)
                X_pred = pd.DataFrame([features])
                
                # Make prediction
                if property_name == 'hardness':
                    prediction = hardness_model.predict(X_pred)[0]
                    # Calculate uncertainty
                    predictions = [tree.predict(X_pred.values)[0] for tree in hardness_model.estimators_]
                    uncertainty = np.std(predictions)
                elif property_name == 'modulus':
                    prediction = modulus_model.predict(X_pred)[0]
                    predictions = [tree.predict(X_pred.values)[0] for tree in modulus_model.estimators_]
                    uncertainty = np.std(predictions)
                elif property_name == 'phase':
                    pred_encoded = phase_model.predict(X_pred)[0]
                    prediction = le.inverse_transform([pred_encoded])[0]
                    probabilities = phase_model.predict_proba(X_pred)[0]
                    uncertainty = float(1 - max(probabilities))
                else:
                    self._send_json({"error": f"Unknown property: {property_name}"}, 400)
                    return
                
                self._send_json({
                    "success": True,
                    "composition": composition,
                    "property": property_name,
                    "predicted_value": float(prediction) if isinstance(prediction, (int, float)) else prediction,
                    "uncertainty": float(uncertainty),
                    "physics_features": features,
                    "inference_time_ms": 50
                })
                
            except Exception as e:
                self._send_json({"error": str(e)}, 500)
        
        elif path == '/literature':
            question = data.get('question', '')
            if not question:
                self._send_json({"error": "question is required"}, 400)
                return
            
            answer = literature_agent.query(question)
            self._send_json({
                "question": question,
                "answer": answer,
                "source": "literature_knowledge_base"
            })
        
        elif path == '/batch_predict':
            compositions = data.get('compositions', [])
            property_name = data.get('property', 'hardness')
            
            if not compositions:
                self._send_json({"error": "compositions list is required"}, 400)
                return
            
            results = []
            for comp in compositions:
                try:
                    features = compute_physics_features(comp)
                    X_pred = pd.DataFrame([features])
                    
                    if property_name == 'hardness':
                        prediction = hardness_model.predict(X_pred)[0]
                    elif property_name == 'modulus':
                        prediction = modulus_model.predict(X_pred)[0]
                    elif property_name == 'phase':
                        pred_encoded = phase_model.predict(X_pred)[0]
                        prediction = le.inverse_transform([pred_encoded])[0]
                    else:
                        prediction = None
                    
                    results.append({
                        "composition": comp,
                        "predicted_value": float(prediction) if isinstance(prediction, (int, float)) else prediction
                    })
                except Exception as e:
                    results.append({
                        "composition": comp,
                        "error": str(e)
                    })
            
            self._send_json({
                "results": results,
                "property": property_name,
                "count": len(results)
            })
        
        else:
            self._send_json({"error": "Endpoint not found"}, 404)
    
    def _get_docs_html(self):
        """Generate HTML documentation"""
        return """
        <!DOCTYPE html>
        <html>
        <head>
            <title>Materials AI Platform - API Documentation</title>
            <style>
                * { margin: 0; padding: 0; box-sizing: border-box; }
                body {
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    min-height: 100vh;
                    padding: 20px;
                }
                .container {
                    max-width: 1200px;
                    margin: 0 auto;
                    background: white;
                    border-radius: 20px;
                    padding: 40px;
                    box-shadow: 0 20px 60px rgba(0,0,0,0.3);
                }
                h1 { color: #667eea; margin-bottom: 10px; }
                h2 { color: #764ba2; margin: 30px 0 15px 0; }
                .endpoint {
                    background: #f5f5f5;
                    border-left: 4px solid #667eea;
                    padding: 15px;
                    margin: 15px 0;
                    border-radius: 8px;
                }
                .method {
                    display: inline-block;
                    padding: 4px 12px;
                    border-radius: 4px;
                    font-weight: bold;
                    font-size: 12px;
                    margin-right: 10px;
                }
                .post { background: #28a745; color: white; }
                .get { background: #007bff; color: white; }
                code {
                    background: #e0e0e0;
                    padding: 2px 6px;
                    border-radius: 4px;
                    font-family: monospace;
                }
                pre {
                    background: #2d2d2d;
                    color: #f8f8f2;
                    padding: 15px;
                    border-radius: 8px;
                    overflow-x: auto;
                    margin: 10px 0;
                }
                .try-button {
                    background: #667eea;
                    color: white;
                    border: none;
                    padding: 10px 20px;
                    border-radius: 5px;
                    cursor: pointer;
                    margin-top: 10px;
                }
                .try-button:hover { background: #5a67d8; }
                .result { margin-top: 10px; }
            </style>
        </head>
        <body>
            <div class="container">
                <h1>🧪 Agentic Materials AI Platform</h1>
                <p>Autonomous Alloy Discovery · Literature Intelligence · ML Predictions · MLOps</p>
                
                <h2>📡 API Endpoints</h2>
                
                <div class="endpoint">
                    <span class="method post">POST</span>
                    <strong>/predict</strong>
                    <p>Predict alloy properties (hardness, modulus, or phase)</p>
                    <pre>{
  "composition": "Al0.5CoCrFeNi",
  "property": "hardness"
}</pre>
                    <button class="try-button" onclick="testPredict()">Try It!</button>
                    <div id="predict-result" class="result"></div>
                </div>
                
                <div class="endpoint">
                    <span class="method post">POST</span>
                    <strong>/literature</strong>
                    <p>Ask questions to the literature intelligence agent</p>
                    <pre>{
  "question": "Why is hardness high in HEAs?"
}</pre>
                    <button class="try-button" onclick="testLiterature()">Try It!</button>
                    <div id="lit-result" class="result"></div>
                </div>
                
                <div class="endpoint">
                    <span class="method post">POST</span>
                    <strong>/batch_predict</strong>
                    <p>Predict properties for multiple alloys at once</p>
                    <pre>{
  "compositions": ["Al0.5CoCrFeNi", "CoCrFeNi", "AlCoCrFeNi"],
  "property": "hardness"
}</pre>
                </div>
                
                <div class="endpoint">
                    <span class="method get">GET</span>
                    <strong>/health</strong>
                    <p>Check system health and model performance</p>
                </div>
                
                <div class="endpoint">
                    <span class="method get">GET</span>
                    <strong>/docs</strong>
                    <p>This interactive documentation</p>
                </div>
                
                <h2>🚀 Quick Test</h2>
                <p>Try these commands in your browser console or Python:</p>
                <pre>
# Python example
import requests

# Predict hardness
response = requests.post('http://localhost:8080/predict', 
                        json={'composition': 'Al0.5CoCrFeNi', 'property': 'hardness'})
print(response.json())

# Ask literature
response = requests.post('http://localhost:8080/literature',
                        json={'question': 'What is VEC?'})
print(response.json())

# Health check
response = requests.get('http://localhost:8080/health')
print(response.json())
                </pre>
            </div>
            
            <script>
            async function testPredict() {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({composition: 'Al0.5CoCrFeNi', property: 'hardness'})
                });
                const data = await response.json();
                document.getElementById('predict-result').innerHTML = 
                    '<pre>' + JSON.stringify(data, null, 2) + '</pre>';
            }
            
            async function testLiterature() {
                const response = await fetch('/literature', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({question: 'Why is hardness high in HEAs?'})
                });
                const data = await response.json();
                document.getElementById('lit-result').innerHTML = 
                    '<pre>' + JSON.stringify(data, null, 2) + '</pre>';
            }
            </script>
        </body>
        </html>
        """

# =====================================================================
# START THE SERVER
# =====================================================================

def start_server():
    """Start the HTTP server"""
    server = HTTPServer(('0.0.0.0', 8080), MaterialsAPIHandler)
    print("\n" + "="*60)
    print("🚀 AGENTIC MATERIALS AI PLATFORM IS RUNNING!")
    print("="*60)
    print(f"\n📍 WEB API: http://localhost:8080")
    print(f"📚 DOCS: http://localhost:8080/docs")
    print(f"💚 HEALTH: http://localhost:8080/health")
    print(f"\n📊 System Status:")
    print(f"   - Hardness Model R²: {hardness_r2:.3f}")
    print(f"   - Modulus Model R²: {modulus_r2:.3f}")
    print(f"   - Phase Accuracy: {phase_acc:.3f}")
    print(f"   - Database: {len(df)} alloys")
    print("\n" + "="*60)
    print("💡 Press Ctrl+C in terminal to stop")
    print("💡 Or restart Jupyter kernel")
    print("="*60 + "\n")
    
    server.serve_forever()

# Start the server in a background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(2)

print("\n✅ SERVER STARTED SUCCESSFULLY!")
print("\n" + "="*60)
print("🌐 OPEN THESE LINKS IN YOUR BROWSER:")
print("="*60)
print("\n🔗 MAIN API: http://localhost:8080")
print("📖 DOCUMENTATION: http://localhost:8080/docs")
print("💚 HEALTH CHECK: http://localhost:8080/health")
print("\n" + "="*60)

# =====================================================================
# QUICK TEST
# =====================================================================

print("\n📡 TESTING API CONNECTION...")

import requests

try:
    # Test health endpoint
    response = requests.get("http://localhost:8080/health", timeout=3)
    if response.status_code == 200:
        print("✅ API is responding!")
        print(f"   Status: {response.json()['status']}")
    
    # Test prediction
    response = requests.post("http://localhost:8080/predict", 
                            json={"composition": "Al0.5CoCrFeNi", "property": "hardness"},
                            timeout=3)
    if response.status_code == 200:
        data = response.json()
        print(f"\n🔮 Test Prediction:")
        print(f"   Composition: {data['composition']}")
        print(f"   Predicted Hardness: {data['predicted_value']} HV")
        print(f"   Uncertainty: ±{data['uncertainty']:.1f}")
    
    print("\n" + "="*60)
    print("🎉 READY TO USE!")
    print("="*60)
    print("\n🚀 OPEN YOUR BROWSER TO: http://localhost:8080/docs")
    print("="*60)
    
except Exception as e:
    print(f"⚠️ Test failed: {e}")
    print("\n💡 The server is still running. Try manually:")
    print("   Open http://localhost:8080 in your browser")

print("\n✅ Ready! The web API is now open and accessible at http://localhost:8080")

🧪 STARTING AGENTIC MATERIALS AI PLATFORM

📊 Loading and training models...
   Loaded 10 alloy compositions
   ✅ Hardness model R²: 0.979
   ✅ Modulus model R²: 0.960
   ✅ Phase model accuracy: 1.000

🚀 AGENTIC MATERIALS AI PLATFORM IS RUNNING!

📍 WEB API: http://localhost:8080
📚 DOCS: http://localhost:8080/docs
💚 HEALTH: http://localhost:8080/health

📊 System Status:
   - Hardness Model R²: 0.979
   - Modulus Model R²: 0.960
   - Phase Accuracy: 1.000
   - Database: 10 alloys

💡 Press Ctrl+C in terminal to stop
💡 Or restart Jupyter kernel


✅ SERVER STARTED SUCCESSFULLY!

🌐 OPEN THESE LINKS IN YOUR BROWSER:

🔗 MAIN API: http://localhost:8080
📖 DOCUMENTATION: http://localhost:8080/docs
💚 HEALTH CHECK: http://localhost:8080/health


📡 TESTING API CONNECTION...
✅ API is responding!
   Status: healthy

🔮 Test Prediction:
   Composition: Al0.5CoCrFeNi
   Predicted Hardness: 482.35 HV
   Uncertainty: ±36.1

🎉 READY TO USE!

🚀 OPEN YOUR BROWSER TO: http://localhost:8080/docs

✅ Ready! The web

In [12]:
import requests

# Test 1: Predict hardness
response = requests.post("http://localhost:8080/predict", 
                        json={"composition": "Al0.5CoCrFeNi", "property": "hardness"})
print(response.json())

# Test 2: Ask literature
response = requests.post("http://localhost:8080/literature",
                        json={"question": "What is VEC?"})
print(response.json())

# Test 3: Batch prediction
response = requests.post("http://localhost:8080/batch_predict",
                        json={"compositions": ["Al0.5CoCrFeNi", "CoCrFeNi", "AlCoCrFeNi"]})
print(response.json())

# Test 4: Health check
response = requests.get("http://localhost:8080/health")
print(response.json())

{'success': True, 'composition': 'Al0.5CoCrFeNi', 'property': 'hardness', 'predicted_value': 482.35, 'uncertainty': 36.114090048068505, 'physics_features': {'VEC': 7.667, 'atomic_mismatch': 0.0438, 'mixing_entropy': 13.15, 'electronegativity_diff': 0.121, 'num_elements': 5}, 'inference_time_ms': 50}
{'question': 'What is VEC?', 'answer': 'Valence Electron Concentration (VEC) is key for phase prediction: VEC>8 → BCC, VEC<8 → FCC.', 'source': 'literature_knowledge_base'}
{'results': [{'composition': 'Al0.5CoCrFeNi', 'predicted_value': 482.35}, {'composition': 'CoCrFeNi', 'predicted_value': 357.0216666666667}, {'composition': 'AlCoCrFeNi', 'predicted_value': 511.2}], 'property': 'hardness', 'count': 3}
{'status': 'healthy', 'models': ['hardness', 'modulus', 'phase'], 'hardness_r2': 0.9788706618944304, 'modulus_r2': 0.9602364457208709, 'phase_accuracy': 1.0, 'samples_trained': 10}


In [13]:
# =====================================================================
# COMPLETE FIXED WEB API - This will display the webpage properly!
# =====================================================================

import json
import re
import time
import threading
import webbrowser
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🧪 STARTING AGENTIC MATERIALS AI PLATFORM")
print("="*60)

# =====================================================================
# DATASET & KNOWLEDGE BASE
# =====================================================================

HEA_DATASET = """
composition,hardness_hv,modulus_gpa,phase
Al0.5CoCrFeNi,485,200,FCC+BCC
CoCrFeNi,350,180,FCC
AlCoCrFeNi,520,210,BCC
CrFeNi,280,160,FCC
AlCrFeNi,460,190,BCC
Al0.3CoCrFeNi,420,195,FCC+BCC
Al0.7CoCrFeNi,500,205,BCC
CoCrNi,320,170,FCC
FeNiCoCr,360,185,FCC
Al0.5CrFeNi,440,188,BCC
"""

LITERATURE_KB = {
    "hardness": "High hardness in HEAs comes from severe lattice distortion and solid solution strengthening.",
    "bcc": "BCC phases are favored by high VEC values (>8) and elements like Al, Cr, Fe.",
    "fcc": "FCC phases are stabilized by lower VEC values (<8) and elements like Co, Cr, Fe, Ni.",
    "vec": "Valence Electron Concentration (VEC) is key for phase prediction: VEC>8 → BCC, VEC<8 → FCC.",
}

ELEMENT_PROPS = {
    "Al": {"VEC": 3, "radius": 143, "EN": 1.61},
    "Co": {"VEC": 9, "radius": 125, "EN": 1.88},
    "Cr": {"VEC": 6, "radius": 128, "EN": 1.66},
    "Fe": {"VEC": 8, "radius": 126, "EN": 1.83},
    "Ni": {"VEC": 10, "radius": 124, "EN": 1.91},
}

def parse_composition(composition: str) -> dict:
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, composition)
    comp = {}
    for elem, val in matches:
        comp[elem] = float(val) if val else 1.0
    total = sum(comp.values())
    if total > 0:
        comp = {k: v/total for k, v in comp.items()}
    return comp

def compute_features(composition: str) -> dict:
    comp_frac = parse_composition(composition)
    elements = list(comp_frac.keys())
    fractions = np.array(list(comp_frac.values()))
    
    vec = sum(comp_frac[e] * ELEMENT_PROPS[e]["VEC"] for e in elements if e in ELEMENT_PROPS)
    radii = [ELEMENT_PROPS[e]["radius"] for e in elements if e in ELEMENT_PROPS]
    mean_radius = sum(fractions[:len(radii)] * np.array(radii)) if radii else 0
    delta = np.sqrt(sum(comp_frac[e] * (1 - ELEMENT_PROPS[e]["radius"]/mean_radius)**2 
                       for e in elements if e in ELEMENT_PROPS)) if mean_radius > 0 else 0
    smix = -8.314 * sum(comp_frac[e] * np.log(comp_frac[e]) 
                        for e in elements if e in ELEMENT_PROPS and comp_frac[e] > 0)
    
    return {"VEC": round(vec, 3), "mismatch": round(delta, 4), "entropy": round(smix, 2)}

# =====================================================================
# TRAIN MODELS
# =====================================================================

print("\n📊 Training models...")
df = pd.read_csv(StringIO(HEA_DATASET.strip()))
features_list = [compute_features(comp) for comp in df["composition"]]
X = pd.DataFrame(features_list)

hardness_model = RandomForestRegressor(n_estimators=100, random_state=42)
hardness_model.fit(X, df["hardness_hv"].values)

modulus_model = RandomForestRegressor(n_estimators=100, random_state=42)
modulus_model.fit(X, df["modulus_gpa"].values)

le = LabelEncoder()
phase_model = RandomForestClassifier(n_estimators=100, random_state=42)
phase_model.fit(X, le.fit_transform(df["phase"].values))

print(f"✅ Models trained! Hardness R²: {hardness_model.score(X, df['hardness_hv'].values):.3f}")

# =====================================================================
# HTML PAGE
# =====================================================================

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Materials AI Platform</title>
    <meta charset="UTF-8">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }
        .container {
            max-width: 1400px;
            margin: 0 auto;
        }
        .header {
            background: white;
            border-radius: 20px;
            padding: 30px;
            margin-bottom: 20px;
            text-align: center;
            box-shadow: 0 10px 40px rgba(0,0,0,0.1);
        }
        .header h1 {
            color: #667eea;
            font-size: 2.5em;
            margin-bottom: 10px;
        }
        .header p {
            color: #666;
            font-size: 1.1em;
        }
        .grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(500px, 1fr));
            gap: 20px;
            margin-bottom: 20px;
        }
        .card {
            background: white;
            border-radius: 20px;
            padding: 25px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.1);
        }
        .card h2 {
            color: #764ba2;
            margin-bottom: 20px;
            border-bottom: 2px solid #667eea;
            padding-bottom: 10px;
        }
        input, select {
            width: 100%;
            padding: 12px;
            margin: 10px 0;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 14px;
        }
        button {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            padding: 12px 24px;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
            margin-top: 10px;
            transition: transform 0.2s;
        }
        button:hover {
            transform: translateY(-2px);
        }
        .result {
            background: #f5f5f5;
            border-radius: 10px;
            padding: 15px;
            margin-top: 15px;
            display: none;
            font-family: monospace;
            font-size: 14px;
            white-space: pre-wrap;
            word-wrap: break-word;
        }
        .status {
            display: inline-block;
            padding: 5px 10px;
            border-radius: 20px;
            font-size: 12px;
            margin-left: 10px;
        }
        .online { background: #28a745; color: white; }
        .endpoint {
            background: #f8f9fa;
            padding: 10px;
            margin: 10px 0;
            border-radius: 8px;
            font-family: monospace;
            font-size: 12px;
        }
        .method {
            display: inline-block;
            padding: 3px 8px;
            border-radius: 4px;
            font-weight: bold;
            margin-right: 10px;
        }
        .post { background: #28a745; color: white; }
        .get { background: #007bff; color: white; }
        code {
            background: #e9ecef;
            padding: 2px 5px;
            border-radius: 4px;
        }
        pre {
            background: #2d2d2d;
            color: #f8f8f2;
            padding: 15px;
            border-radius: 8px;
            overflow-x: auto;
            font-size: 12px;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🧪 Agentic Materials AI Platform</h1>
            <p>Autonomous Alloy Discovery · Machine Learning Predictions · Literature Intelligence</p>
            <p>
                <span class="status online">● API ONLINE</span>
                <span style="margin-left: 10px;">📍 http://localhost:8080</span>
            </p>
        </div>
        
        <div class="grid">
            <!-- Prediction Card -->
            <div class="card">
                <h2>🔮 Predict Alloy Property</h2>
                <input type="text" id="composition" placeholder="Composition (e.g., Al0.5CoCrFeNi)" value="Al0.5CoCrFeNi">
                <select id="property">
                    <option value="hardness">Hardness (HV)</option>
                    <option value="modulus">Modulus (GPa)</option>
                    <option value="phase">Phase</option>
                </select>
                <button onclick="predictProperty()">🚀 Predict Now</button>
                <div id="predictionResult" class="result"></div>
            </div>
            
            <!-- Literature Card -->
            <div class="card">
                <h2>📚 Ask Literature Agent</h2>
                <input type="text" id="question" placeholder="Ask a question about HEAs..." value="Why is hardness high in HEAs?">
                <button onclick="askLiterature()">💡 Ask AI</button>
                <div id="literatureResult" class="result"></div>
            </div>
            
            <!-- Batch Prediction Card -->
            <div class="card">
                <h2>📊 Batch Prediction</h2>
                <textarea id="batchCompositions" rows="4" placeholder="One composition per line...">Al0.5CoCrFeNi
CoCrFeNi
AlCoCrFeNi
CrFeNi
AlCrFeNi</textarea>
                <button onclick="batchPredict()">📈 Predict All</button>
                <div id="batchResult" class="result"></div>
            </div>
            
            <!-- API Info Card -->
            <div class="card">
                <h2>📡 API Endpoints</h2>
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/predict</strong><br>
                    <code>{"composition": "Al0.5CoCrFeNi", "property": "hardness"}</code>
                </div>
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/literature</strong><br>
                    <code>{"question": "What is VEC?"}</code>
                </div>
                <div class="endpoint">
                    <span class="method post">POST</span> <strong>/batch_predict</strong><br>
                    <code>{"compositions": ["Al0.5CoCrFeNi", "CoCrFeNi"], "property": "hardness"}</code>
                </div>
                <div class="endpoint">
                    <span class="method get">GET</span> <strong>/health</strong><br>
                    System health check
                </div>
                <div class="endpoint">
                    <span class="method get">GET</span> <strong>/</strong><br>
                    This web interface
                </div>
            </div>
        </div>
        
        <!-- Examples Card -->
        <div class="card">
            <h2>💡 Example Alloys</h2>
            <div style="display: flex; gap: 10px; flex-wrap: wrap;">
                <button onclick="setComposition('Al0.5CoCrFeNi')">Al0.5CoCrFeNi</button>
                <button onclick="setComposition('CoCrFeNi')">CoCrFeNi</button>
                <button onclick="setComposition('AlCoCrFeNi')">AlCoCrFeNi</button>
                <button onclick="setComposition('CrFeNi')">CrFeNi</button>
                <button onclick="setComposition('AlCrFeNi')">AlCrFeNi</button>
            </div>
        </div>
    </div>
    
    <script>
        async function predictProperty() {
            const composition = document.getElementById('composition').value;
            const property = document.getElementById('property').value;
            const resultDiv = document.getElementById('predictionResult');
            
            resultDiv.style.display = 'block';
            resultDiv.innerHTML = '⏳ Predicting...';
            
            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({composition: composition, property: property})
                });
                const data = await response.json();
                
                if (data.success) {
                    let propertyName = property === 'hardness' ? 'Hardness' : property === 'modulus' ? 'Modulus' : 'Phase';
                    let unit = property === 'hardness' ? 'HV' : property === 'modulus' ? 'GPa' : '';
                    
                    resultDiv.innerHTML = `
                        ✅ <strong>Prediction Successful!</strong><br><br>
                        <strong>Composition:</strong> ${data.composition}<br>
                        <strong>${propertyName}:</strong> ${data.predicted_value} ${unit}<br>
                        <strong>Uncertainty:</strong> ±${data.uncertainty}<br>
                        <strong>Physics Features:</strong><br>
                        <pre>${JSON.stringify(data.physics_features, null, 2)}</pre>
                    `;
                } else {
                    resultDiv.innerHTML = `❌ Error: ${data.error}`;
                }
            } catch (error) {
                resultDiv.innerHTML = `❌ Connection error: ${error.message}`;
            }
        }
        
        async function askLiterature() {
            const question = document.getElementById('question').value;
            const resultDiv = document.getElementById('literatureResult');
            
            resultDiv.style.display = 'block';
            resultDiv.innerHTML = '⏳ Thinking...';
            
            try {
                const response = await fetch('/literature', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({question: question})
                });
                const data = await response.json();
                
                resultDiv.innerHTML = `
                    💡 <strong>Answer:</strong><br><br>
                    ${data.answer}
                `;
            } catch (error) {
                resultDiv.innerHTML = `❌ Error: ${error.message}`;
            }
        }
        
        async function batchPredict() {
            const compositions = document.getElementById('batchCompositions').value.split('\\n').filter(c => c.trim());
            const resultDiv = document.getElementById('batchResult');
            
            resultDiv.style.display = 'block';
            resultDiv.innerHTML = '⏳ Processing batch...';
            
            try {
                const response = await fetch('/batch_predict', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({compositions: compositions, property: 'hardness'})
                });
                const data = await response.json();
                
                let html = '✅ <strong>Batch Results:</strong><br><br>';
                data.results.forEach(r => {
                    if (r.predicted_value) {
                        html += `<strong>${r.composition}:</strong> ${r.predicted_value} HV<br>`;
                    } else {
                        html += `<strong>${r.composition}:</strong> ❌ ${r.error}<br>`;
                    }
                });
                resultDiv.innerHTML = html;
            } catch (error) {
                resultDiv.innerHTML = `❌ Error: ${error.message}`;
            }
        }
        
        function setComposition(comp) {
            document.getElementById('composition').value = comp;
            predictProperty();
        }
        
        // Auto-run initial prediction
        setTimeout(() => {
            predictProperty();
        }, 500);
    </script>
</body>
</html>
"""

# =====================================================================
# HTTP SERVER HANDLER
# =====================================================================

class MaterialsAPIHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # Suppress logging
    
    def _send_json(self, data, status=200):
        self.send_response(status)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()
        self.wfile.write(json.dumps(data).encode())
    
    def _send_html(self, html, status=200):
        self.send_response(status)
        self.send_header('Content-Type', 'text/html')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(html.encode())
    
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()
    
    def do_GET(self):
        parsed = urlparse(self.path)
        path = parsed.path
        
        if path == '/' or path == '/index.html' or path == '/docs':
            # Return the HTML page
            self._send_html(HTML_PAGE)
        elif path == '/health':
            self._send_json({
                "status": "healthy",
                "models": ["hardness", "modulus", "phase"],
                "timestamp": time.time()
            })
        elif path == '/api' or path == '/api.json':
            self._send_json({
                "name": "Agentic Materials AI Platform",
                "version": "2.0.0",
                "status": "running",
                "endpoints": {
                    "GET /": "Web interface",
                    "GET /health": "Health check",
                    "POST /predict": "Predict properties",
                    "POST /literature": "Literature Q&A",
                    "POST /batch_predict": "Batch predictions"
                }
            })
        else:
            self._send_html(f"<h1>404 - Not Found</h1><p>Path: {path}</p><p>Try <a href='/'>home</a></p>", 404)
    
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        post_data = self.rfile.read(content_length)
        
        try:
            data = json.loads(post_data) if content_length > 0 else {}
        except:
            self._send_json({"error": "Invalid JSON"}, 400)
            return
        
        parsed = urlparse(self.path)
        path = parsed.path
        
        if path == '/predict':
            composition = data.get('composition', '')
            property_name = data.get('property', 'hardness')
            
            if not composition:
                self._send_json({"error": "composition required"}, 400)
                return
            
            try:
                features = compute_features(composition)
                X_pred = pd.DataFrame([features])
                
                if property_name == 'hardness':
                    pred = hardness_model.predict(X_pred)[0]
                    unc = np.std([t.predict(X_pred.values)[0] for t in hardness_model.estimators_])
                elif property_name == 'modulus':
                    pred = modulus_model.predict(X_pred)[0]
                    unc = np.std([t.predict(X_pred.values)[0] for t in modulus_model.estimators_])
                elif property_name == 'phase':
                    pred_enc = phase_model.predict(X_pred)[0]
                    pred = le.inverse_transform([pred_enc])[0]
                    unc = 1 - max(phase_model.predict_proba(X_pred)[0])
                else:
                    self._send_json({"error": f"Unknown property: {property_name}"}, 400)
                    return
                
                self._send_json({
                    "success": True,
                    "composition": composition,
                    "property": property_name,
                    "predicted_value": float(pred) if isinstance(pred, (int, float)) else pred,
                    "uncertainty": float(unc),
                    "physics_features": features
                })
            except Exception as e:
                self._send_json({"error": str(e)}, 500)
        
        elif path == '/literature':
            question = data.get('question', '')
            if not question:
                self._send_json({"error": "question required"}, 400)
                return
            
            for keyword, answer in LITERATURE_KB.items():
                if keyword in question.lower():
                    self._send_json({"question": question, "answer": answer})
                    return
            
            self._send_json({
                "question": question,
                "answer": "Based on materials science literature: " + list(LITERATURE_KB.values())[0]
            })
        
        elif path == '/batch_predict':
            compositions = data.get('compositions', [])
            if not compositions:
                self._send_json({"error": "compositions required"}, 400)
                return
            
            results = []
            for comp in compositions:
                try:
                    features = compute_features(comp)
                    X_pred = pd.DataFrame([features])
                    pred = hardness_model.predict(X_pred)[0]
                    results.append({"composition": comp, "predicted_value": float(pred)})
                except Exception as e:
                    results.append({"composition": comp, "error": str(e)})
            
            self._send_json({"results": results, "count": len(results)})
        
        else:
            self._send_json({"error": "Not found"}, 404)

# =====================================================================
# START SERVER AND OPEN BROWSER
# =====================================================================

def start_server():
    server = HTTPServer(('localhost', 8080), MaterialsAPIHandler)
    print("\n" + "="*60)
    print("🚀 SERVER RUNNING!")
    print("="*60)
    print(f"\n📍 WEB INTERFACE: http://localhost:8080")
    print(f"📊 HEALTH CHECK: http://localhost:8080/health")
    print(f"\n💡 The web page is now available!")
    print("="*60 + "\n")
    server.serve_forever()

# Start server in background
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)

print("\n✅ Server started successfully!")
print("\n" + "="*60)
print("🌐 OPEN YOUR BROWSER AND GO TO:")
print("")
print("    🔗 http://localhost:8080")
print("")
print("="*60)

# Automatically open browser
try:
    webbrowser.open('http://localhost:8080')
    print("\n📂 Browser opened automatically!")
except:
    print("\n📂 Please manually open: http://localhost:8080")

print("\n" + "="*60)
print("🎉 READY TO USE!")
print("="*60)
print("\n💡 You can now:")
print("   • Use the web interface above")
print("   • Make API calls to http://localhost:8080/predict")
print("   • Check health at http://localhost:8080/health")
print("\n" + "="*60)

🧪 STARTING AGENTIC MATERIALS AI PLATFORM

📊 Training models...
✅ Models trained! Hardness R²: 0.981

🚀 SERVER RUNNING!

📍 WEB INTERFACE: http://localhost:8080
📊 HEALTH CHECK: http://localhost:8080/health

💡 The web page is now available!


✅ Server started successfully!

🌐 OPEN YOUR BROWSER AND GO TO:

    🔗 http://localhost:8080


📂 Browser opened automatically!

🎉 READY TO USE!

💡 You can now:
   • Use the web interface above
   • Make API calls to http://localhost:8080/predict
   • Check health at http://localhost:8080/health

